# Riset_MARL — Notebook Master Penelitian Tesis

**Integrasi Deep Knowledge Tracing dan Multi-Agent Reinforcement Learning untuk Adaptive Testing Berbasis Diagnosis Kognitif Mahasiswa**

Dataset: **DBE-KT22 asli** · Encoder: **DKT berbasis GRU** · Pembanding: **RL-Flat**, **RL-Feat** · Metode utama: **MARL (Planning, KC Selection, Item Selection)**.

Notebook ini merupakan **penelitian ulang dari awal**, bukan kelanjutan checkpoint penelitian sebelumnya. DKT dilatih ulang menggunakan mahasiswa training, kebijakan RL juga dilatih ulang, dan output disimpan terpisah. Arsitektur MARL mempertahankan *planning → KC → item* dengan fitur diagnostik per soal serta reward ranking-aware. **Tidak ada jaminan MARL unggul pada semua metrik**; hasil harus dicatat apa adanya.

### Daftar isi
1. Persiapan lingkungan dan konfigurasi
2. Dataset DBE-KT22 serta pemisahan mahasiswa
3. DKT: arsitektur, training, dan diagnostik
4. Simulasi CAT offline dan reward
5. RL-Flat, RL-Feat, dan tiga agen MARL
6. PPO: training, checkpoint, dan resume
7. Eksperimen training dan evaluasi development
8. Grafik, tabel, dan laporan
9. Multi-seed
10. Ablation study
11. Pembekuan model (*freeze*)
12. Final test terkunci dan statistik
13. Laporan hasil terintegrasi (tanpa training ulang)
14. Pelaporan publik berbasis hasil agregat (tanpa dataset privat)

**Penyajian hasil tanpa training:** atur `MODE="report"` (default pada versi ini), lalu jalankan **Runtime → Restart session and run all**. Mode ini hanya membaca CSV, gambar, dan manifest yang sudah tersimpan; checkpoint dan prediksi tidak diubah. Hasil laporan disimpan di `outputs/laporan/`.

**Cara pakai eksperimen:** jalankan `MODE='smoke'` dahulu, kemudian `full`, `multiseed`, `ablation`, `freeze`, dan terakhir `final`. Pada setiap perubahan mode, pilih **Runtime → Restart session and run all**. Mode `final` hanya dijalankan sekali ketika model sudah dibekukan.

**Publikasi GitHub:** pengunjung repositori dapat memilih `MODE="public"` di sel *SETUP* untuk menampilkan ulang hasil agregat dan grafik tanpa mengakses data mahasiswa, checkpoint, atau Google Drive. Mode `report` tetap digunakan untuk arsip penelitian pribadi di Google Drive. Rincian reproduksi, metadata, dan keterbatasan ada pada `README.md` dan folder `docs/`.


## 0.1 Petunjuk pembaca dan batas penggunaan ilmiah

Notebook ini mendukung dua kebutuhan: **reproduksi tampilan hasil agregat** (`public`, tanpa Google Drive) dan **pelaporan tesis pribadi** (`report`, dengan artefak penelitian di Drive). Mode `smoke`, `full`, `multiseed`, dan `ablation` dipakai untuk eksperimen yang memerlukan dataset dan komputasi training. `freeze` dan `final` terpisah dari pelatihan.

**Batas interpretasi:** penelitian mengevaluasi *offline replay* DBE-KT22. Tiga kebijakan MARL adalah aktor hierarkis kooperatif dengan satu objective PPO dan critic bersama; implementasi ini tidak otomatis setara dengan MAPPO formal. Kelompok test historis pernah digunakan pada penelitian sebelumnya sehingga tidak dapat diklaim sebagai holdout independen baru.

Notebook publik menampilkan grafik dan tabel dari artefak agregat; ia **tidak** melakukan perhitungan ulang prediksi mahasiswa dari model. Perbedaan hasil numerik dan bukti signifikansi dijelaskan pada `docs/STATISTIK_DAN_KLAIM.md`.

# 1. Persiapan lingkungan dan konfigurasi

## 1.1 Tujuan dan protokol komputasi

Semua model dilatih menggunakan DKT, lingkungan CAT, batas 20 soal, dan reward yang sama. Perbedaannya adalah *policy representation*: RL-Flat memetakan state langsung ke aksi soal; RL-Feat menggunakan state dan fitur soal; MARL memilih strategi, KC, dan soal secara berurutan.

**Catatan tentang penelitian ulang:** data final test penelitian sebelumnya sudah pernah dilihat. Walaupun notebook ini menjaga kelompok test tidak dipakai untuk tuning, kelompok tersebut tidak lagi merupakan holdout yang sepenuhnya baru secara sejarah penelitian. Laporkan keterbatasan ini jika memakai dataset yang sama.

## 1.2 Membuat folder Google Drive baru (jalankan sebelum mengunggah dataset)

Notebook ini **tidak menggunakan folder maupun checkpoint penelitian lama**. Folder utama baru bernama **`MyDrive/Riset_MARL/`**. Struktur dibuat secara otomatis ketika sel persiapan di bawah dijalankan.

**Urutan pertama kali:**

1. Buka notebook ini di Google Colab dan jalankan **hanya sel `SETUP GOOGLE DRIVE` di bawah**, bukan `Run all` dulu.
2. Berikan izin *mount* Google Drive jika diminta. Pastikan output menampilkan `MyDrive/Riset_MARL/input/`.
3. Buka [Google Drive](https://drive.google.com/drive/my-drive) → `Drive Saya` → `Riset_MARL` → `input`.
4. Klik **Baru → Upload file**, lalu unggah **satu berkas ZIP dataset DBE-KT22 asli**. Jangan ekstrak ZIP, jangan unggah notebook/repository/kumpulan file CSV terpisah ke folder `input`.
5. ZIP yang benar memuat arsip di dalamnya yang namanya berakhiran **`datafiles_100102_csv.zip`**, berisi `Transaction.csv`, `Questions.csv`, dan `Question_KC_Relationships.csv`.
6. Setelah unggahan **100% selesai**, kembali ke Colab lalu pilih **Runtime → Restart session and run all**. Dengan `MODE='smoke'`, semua proses dimulai dari awal.

**Penting:** Folder baru dibuat pada Google Drive Anda oleh Colab setelah Anda menjalankan sel di bawah. Saya tidak bisa membuat folder secara langsung di akun Anda dari percakapan ini. Jangan memindahkan dataset selama eksperimen berjalan.


In [ ]:
# SETUP — pilih MODE di sini. Mode public tidak perlu Google Drive atau dataset asli.
import sys, os
from pathlib import Path

# Penggunaan pribadi: report, smoke, full, multiseed, ablation, freeze, final.
# Pelaporan dari repositori GitHub tanpa data privat: public.
MODE = 'kc_smoke'  # Setelah lolos, ubah ke 'kc_audit' untuk lima seed.

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass

if MODE == 'public':
    # Repository harus tersedia secara lokal, misalnya /content/Riset_MARL setelah git clone.
    repo_hint = os.environ.get('RISET_MARL_REPO_DIR')
    ROOT = Path(repo_hint).expanduser().resolve() if repo_hint else (
        Path('/content/Riset_MARL') if Path('/content/Riset_MARL/results/public').is_dir()
        else Path.cwd().resolve())
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Riset_MARL')
else:
    # Jalur lokal hanya untuk pemeriksaan kode di luar Google Colab.
    ROOT = (Path('/mnt/data/Riset_MARL') if Path('/mnt/data').exists() else Path.cwd() / 'Riset_MARL')

INPUT = ROOT / 'input'
WORKSPACE = ROOT / ('outputs_kc_comparison' if MODE in ('kc_smoke', 'kc_audit') else 'outputs')
for folder in (ROOT, INPUT, WORKSPACE):
    folder.mkdir(parents=True, exist_ok=True)

print('DIREKTORI PUBLIKASI SIAP' if MODE == 'public' else 'FOLDER GOOGLE DRIVE BERHASIL DISIAPKAN')
print('Folder utama:', ROOT)
print('UNGGAH ZIP DATASET KE:', INPUT)
print('Folder output:', WORKSPACE)
zips = [] if MODE == 'public' else sorted(INPUT.glob('*.zip'))
if MODE == 'public':
    print('MODE PUBLIC: dataset mentah, Drive dan checkpoint tidak dibutuhkan.')
    print('Sumber publik:', ROOT / 'results' / 'public')
elif len(zips) == 0:
    print('STATUS: dataset belum diunggah. Upload 1 ZIP asli DBE-KT22, lalu jalankan ulang notebook dari awal.')
elif len(zips) == 1:
    print('STATUS: ditemukan 1 ZIP:', zips[0].name)
else:
    print('PERINGATAN: ada', len(zips), 'ZIP di input. Sisakan hanya 1 ZIP dataset asli DBE-KT22.')


In [ ]:
import os, sys, json, zipfile, random, copy, warnings, io, hashlib, shutil, time
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
from scipy.special import expit
from IPython.display import display

# MODE ditetapkan hanya pada sel SETUP di atas; jangan didefinisikan ulang.
SEED = 42
MULTI_SEEDS = [42, 123, 321, 2026, 2027]
ABLATION_SEEDS = [42, 123]
FINAL_TEST_UNLOCK = False  # Ubah True hanya sesudah mode freeze berhasil.
UPLOAD_DARI_LAPTOP = False
assert MODE in ('public', 'report', 'smoke', 'full', 'multiseed', 'ablation', 'freeze', 'final', 'kc_smoke', 'kc_audit')

# ROOT/INPUT/WORKSPACE telah dibuat di sel SETUP GOOGLE DRIVE.
# Jalur ini terpisah sepenuhnya dari eksperimen terdahulu.
if 'ROOT' not in globals() or (MODE != 'public' and ROOT.name != 'Riset_MARL'):
    raise RuntimeError('Jalankan sel SETUP GOOGLE DRIVE lebih dahulu.')
INPUT = ROOT/'input'
MODE_ROOT = WORKSPACE/'laporan' if MODE == 'report' else WORKSPACE/'laporan_public' if MODE == 'public' else WORKSPACE/'runs'/MODE
MODE_ROOT.mkdir(parents=True, exist_ok=True)
INPUT.mkdir(parents=True, exist_ok=True)
REPORTS = MODE_ROOT/'reports'
TABLES = MODE_ROOT/'tables'
FIGURES = MODE_ROOT/'figures'
for p in (REPORTS, TABLES, FIGURES): p.mkdir(parents=True, exist_ok=True)
DKT_SCOPE = 'smoke' if MODE in ('smoke', 'kc_smoke') else 'full'
DKT_DIR = WORKSPACE/'dkt'/DKT_SCOPE
DKT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    seed:int = SEED
    split_seed:int = 20261115
    test_length:int = 20
    target_frac:float = .20
    hidden:int = 48
    emb:int = 32
    dkt_lr:float = .002
    ppo_lr:float = .0003
    gamma:float = .98
    gae_lambda:float = .95
    clip:float = .20
    reward_coverage:float = 1.0
    reward_bce_gain:float = 1.5
    reward_rank:float = .8
    reward_information:float = .07
    ppo_epochs:int = 2
    horizon:int = 20
    dkt_epochs:int = 15
    updates:int = 50
    episodes_update:int = 20
    validation_interval_full:int = 5
    bootstrap_n:int = 2000
    shared_prior:bool = True
    shared_prior_entropy:float = -1.0
    shared_prior_difficulty:float = .5
    dkt_validation_fraction:float = .20
    policy_selection_fraction_full:float = .60
    difficulty_source:str = 'train_only'
    smoke:bool = False
    train_students:int = 0
    val_students:int = 0

C = Config()
C.smoke = MODE in ('smoke', 'kc_smoke')
if C.smoke:
    C.dkt_epochs = 2
    C.updates = 2
    C.episodes_update = 2
    C.ppo_epochs = 1
    C.train_students = 140
    C.val_students = 24
    C.bootstrap_n = 300

DEVICE = torch.device('cpu')  # Interaksi PPO per soal lebih cocok untuk CPU.
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:torch.set_num_threads(1)
    except RuntimeError:pass
seed_everything(C.seed)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.partial')
    tmp.write_text(json.dumps(obj,indent=2,default=str,ensure_ascii=False),encoding='utf8')
    os.replace(tmp,path)

def save_torch(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.partial')
    torch.save(obj,tmp);os.replace(tmp,path)

if IN_COLAB and UPLOAD_DARI_LAPTOP:
    from google.colab import files
    for name,data in files.upload().items():
        if not name.lower().endswith('.zip'):raise ValueError('Hanya ZIP asli DBE-KT22')
        (INPUT/name).write_bytes(data)

print('MODE:',MODE,' | CPU threads:',torch.get_num_threads())
print('Lokasi dataset:',INPUT)
print('Lokasi hasil:',WORKSPACE)
print('DKT:',DKT_DIR)
if MODE in ('report', 'public'): print('MODE LAPORAN: tidak menjalankan training, inferensi ataupun evaluasi test.')
print('Final test unlocked:',FINAL_TEST_UNLOCK)

## 1.3 Konfigurasi yang tidak boleh dituning menggunakan final test

| Parameter | Smoke | Full/multi-seed |
|---|---:|---:|
| DKT epochs | 2 | 15 |
| PPO updates/model | 2 | 50 |
| Episode/update | 2 | 20 |
| PPO epoch/update | 1 | 2 |
| Policy checkpoint selection | 24 mahasiswa | 74 mahasiswa |
| Audit independen dari pemilihan checkpoint | 100 mahasiswa (20 digunakan untuk laporan ringkas) | 50 mahasiswa |

Output: `MyDrive/Riset_MARL/outputs/`. Pemisahan mode mencegah hasil smoke menimpa checkpoint full.

# 2. DBE-KT22: audit dan pemisahan mahasiswa

## 2.1 Membaca ZIP asli, interaksi kronologis, relasi soal–KC

Semua interaksi diurutkan menurut `student_id`, `start_time`, dan `id`. Duplikasi respons untuk soal yang sama oleh satu mahasiswa dibuang dengan menyimpan respons pertama. Hanya mahasiswa dengan minimal 28 interaksi unik yang masuk eksperimen.

**Penting:** pengujian CAT dilakukan sebagai replay offline. Soal yang pernah dikerjakan mahasiswa menjadi kumpulan aksi, bukan bank soal interaktif yang mempunyai label jawaban potensial untuk semua soal. Ini membatasi interpretasi generalisasi ke CAT dunia nyata.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    zip_candidates = sorted(INPUT.glob('*.zip'))
    if len(zip_candidates) == 0:
        raise FileNotFoundError(
            f'Dataset belum tersedia. Buka Google Drive > MyDrive > Riset_MARL > input, '
            f'unggah 1 ZIP ASLI DBE-KT22 (jangan diekstrak), '
            f'kemudian jalankan Run all kembali. Folder: {INPUT}'
        )
    if len(zip_candidates) > 1:
        raise ValueError(f'Folder input harus berisi TEPAT SATU ZIP dataset asli; ditemukan {len(zip_candidates)}: {zip_candidates}')
    dataset_zip = zip_candidates[0]
    print('Sumber dataset:',dataset_zip)
    print('SHA256:',hashlib.sha256(dataset_zip.read_bytes()).hexdigest())
    with zipfile.ZipFile(dataset_zip) as outer:
        nested = [n for n in outer.namelist() if n.endswith('datafiles_100102_csv.zip')]
        if len(nested)!=1:raise ValueError('Arsip bukan ZIP DBE-KT22 asli (datafiles_100102_csv.zip tidak ditemukan)')
        with zipfile.ZipFile(io.BytesIO(outer.read(nested[0]))) as datazip:
            def getcsv(name): return pd.read_csv(io.BytesIO(datazip.read(name)),low_memory=False)
            tx = getcsv('Transaction.csv')
            relation = getcsv('Question_KC_Relationships.csv')
            question_info = getcsv('Questions.csv')[['id','difficulty']]  # metadata audit saja, tidak dipakai sebagai difficulty
    required = {'student_id','question_id','start_time','answer_state','id'}
    assert required <= set(tx.columns)
    tx['start_time'] = pd.to_datetime(tx['start_time'],utc=True,errors='coerce')
    tx = tx.dropna(subset=['student_id','question_id','start_time','answer_state']).copy()
    tx = tx.sort_values(['student_id','start_time','id'],kind='stable')
    tx = tx.drop_duplicates(['student_id','question_id'],keep='first')
    question_ids = sorted(set(relation.question_id.astype(int)) & set(tx.question_id.astype(int)))
    k_raw = sorted(set(relation.loc[relation.question_id.isin(question_ids),'knowledgecomponent_id'].astype(int)))
    q_to_i = {q:i for i,q in enumerate(question_ids)}
    k_to_i = {k:i for i,k in enumerate(k_raw)}
    Q,K = len(question_ids),len(k_raw)
    qkc = np.zeros((Q,K),dtype=np.float32)
    for row in relation.itertuples(index=False):
        if int(row.question_id) in q_to_i and int(row.knowledgecomponent_id) in k_to_i:
            qkc[q_to_i[int(row.question_id)],k_to_i[int(row.knowledgecomponent_id)]]=1
    qdiff=np.zeros(Q,dtype=np.float32)  # diisi dari data train-only setelah split
    users={}
    for sid,df in tx.groupby('student_id',sort=False):
        seq=[(q_to_i[int(q)],int(y)) for q,y in zip(df.question_id,df.answer_state) if int(q) in q_to_i]
        if len(seq)>=C.test_length+8:users[str(sid)]=seq
    assert users and np.all(qkc.sum(axis=1)>0)
    print(f'Jumlah pengguna layak={len(users)}, soal={Q}, KC={K}, respons unik={sum(map(len,users.values()))}')
    print('Kelas positif:',round(np.mean([y for seq in users.values() for _,y in seq]),4))
    print('Rentang panjang:',min(map(len,users.values())),max(map(len,users.values())))
    # Split berdasarkan mahasiswa disjoint: train / validation / locked test.
    rng = np.random.default_rng(C.split_seed)
    all_sids = np.array(sorted(users)); rng.shuffle(all_sids)
    n_train, n_val = int(0.70*len(all_sids)), int(0.15*len(all_sids))
    SPLIT = {'train': all_sids[:n_train].tolist(),
             'val': all_sids[n_train:n_train+n_val].tolist(),
             'test': all_sids[n_train+n_val:].tolist()}
    assert not(set(SPLIT['train']) & set(SPLIT['val']))
    assert not(set(SPLIT['train']) & set(SPLIT['test']))
    assert not(set(SPLIT['val']) & set(SPLIT['test']))

    # Validation dibagi TANPA test: DKT checkpoint, policy checkpoint, policy audit.
    # Walaupun hasil v2 telah dilihat, protokol v3 tidak memanfaatkan target 'test'.
    n_dkt_val=max(10,int(round(len(SPLIT['val'])*C.dkt_validation_fraction)))
    DKT_VAL_IDS=SPLIT['val'][:n_dkt_val]
    POLICY_VAL_IDS=SPLIT['val'][n_dkt_val:]
    n_policy_select=(C.val_students if C.val_students else
                     max(12,int(round(len(POLICY_VAL_IDS)*C.policy_selection_fraction_full))))
    assert 0<n_policy_select<len(POLICY_VAL_IDS)
    POLICY_SELECT_IDS=POLICY_VAL_IDS[:n_policy_select]
    POLICY_AUDIT_IDS=POLICY_VAL_IDS[n_policy_select:]
    assert not(set(DKT_VAL_IDS) & set(POLICY_SELECT_IDS))
    assert not(set(DKT_VAL_IDS) & set(POLICY_AUDIT_IDS))
    assert not(set(POLICY_SELECT_IDS) & set(POLICY_AUDIT_IDS))

## 2.2 Pembagian mahasiswa dan rancangan replay

Pemisahan menggunakan **student ID**, bukan baris respons. Kelompok validation dibagi menjadi: DKT validation, policy selection, dan policy audit. Untuk setiap mahasiswa, 20% interaksi terakhir menjadi **target diagnosis** yang tidak tersedia untuk aksi agen. Bagian sebelumnya menjadi kumpulan soal yang dapat direplay.

**Difficulty soal** dihitung dari hasil interaksi **mahasiswa training saja** (Beta(2,2) smoothing); metadata `Questions.csv` tetap dibaca untuk audit tetapi tidak masuk fitur model. Ini lebih ketat daripada smoke kandidat terdahulu, sehingga hasil angka sebelumnya tidak dijamin sama.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Last 20% chronological logs are target prediction; prefix is offline CAT action pool.
    def student_case(sid):
        seq=users[sid];n_target=max(8,int(round(C.target_frac*len(seq))))
        target,active=seq[-n_target:],seq[:-n_target]
        assert len(active)>=C.test_length and len(target)>=8
        assert set(q for q,_ in active).isdisjoint(set(q for q,_ in target))
        return {'sid':sid,'active':active,'target':target}

    CASES={split:[student_case(sid) for sid in ids] for split,ids in SPLIT.items()}
    CASES['policy_select']=[student_case(s) for s in POLICY_SELECT_IDS]
    CASES['policy_audit']=[student_case(s) for s in POLICY_AUDIT_IDS]
    CASES['dkt_val']=[student_case(s) for s in DKT_VAL_IDS]
    CASE_LOOKUP={c['sid']:c for v in CASES.values() for c in v}

    assert not(set(DKT_VAL_IDS) & set(c['sid'] for c in CASES['policy_select']))
    save_json(TABLES/'split_manifest_private.json', {
        'split_seed':C.split_seed,'student_ids':SPLIT,
        'dkt_validation_ids':DKT_VAL_IDS,'ppo_selection_ids':POLICY_SELECT_IDS,
        'ppo_audit_ids':POLICY_AUDIT_IDS,
        'target_fraction':C.target_frac,'length':C.test_length,
        'note':'validation subgroups are disjoint from each other and held-out test'})
    print('Jumlah train/val/test:',{s:len(SPLIT[s]) for s in SPLIT})
    print('DKT-val / PPO-select / PPO-audit:',len(DKT_VAL_IDS),len(POLICY_SELECT_IDS),len(POLICY_AUDIT_IDS))
    print('Minimal active=',min(len(c['active']) for v in CASES.values() for c in v))

    # Item difficulty train-only, dihitung hanya dari prefix mahasiswa training.
    correct = np.zeros(Q, dtype=np.float64)
    num = np.zeros(Q, dtype=np.float64)
    for sid in SPLIT['train']:
        for question, answer in CASE_LOOKUP[sid]['active']:
            correct[question] += float(answer)
            num[question] += 1.0
    qdiff[:] = ((num - correct) + 2.0) / (num + 4.0)
    assert (qdiff >= 0).all() and (qdiff <= 1).all()
    print('Kesulitan train-only (min / maks):',round(float(qdiff.min()),3),round(float(qdiff.max()),3))
    print('Old held-out test tetap TERKUNCI untuk penelitian ulang ini:',len(CASES['test']),'mahasiswa')

## 2.3 Validasi protokol data

Simpan manifest dataset dan split; pastikan tidak ada mahasiswa yang muncul dalam lebih dari satu kelompok. Semua pipeline pelatihan di bawah dibatasi pada `train` untuk update parameter dan `policy_select` untuk pemilihan checkpoint PPO. `policy_audit` hanya untuk laporan diagnostik.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    DATASET_SHA = sha256(dataset_zip)
    SPLIT_PUBLIC = {k:len(v) for k,v in SPLIT.items()}
    SPLIT_PUBLIC.update({'dkt_val':len(DKT_VAL_IDS),'policy_select':len(POLICY_SELECT_IDS),'policy_audit':len(POLICY_AUDIT_IDS)})
    SPLIT_SHA = hashlib.sha256(json.dumps({'SPLIT':SPLIT,'DKT_VAL_IDS':DKT_VAL_IDS,
        'POLICY_SELECT_IDS':POLICY_SELECT_IDS,'POLICY_AUDIT_IDS':POLICY_AUDIT_IDS},sort_keys=True).encode()).hexdigest()
    DIFFICULTY_SHA=hashlib.sha256(qdiff.tobytes()).hexdigest()
    assert len(set(SPLIT['train'])&set(SPLIT['test']))==0
    assert len(set(POLICY_SELECT_IDS)&set(POLICY_AUDIT_IDS))==0
    assert all(len(c['active'])>=C.test_length for group in CASES.values() for c in group)
    assert all(set(q for q,_ in c['active']).isdisjoint(q for q,_ in c['target']) for group in CASES.values() for c in group)
    meta={'dataset_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'difficulty_sha256':DIFFICULTY_SHA,
          'students':SPLIT_PUBLIC,'questions':Q,'knowledge_components':K,
          'config':asdict(C),'difficulty_source':C.difficulty_source,'mode':MODE,
          'offline_replay':True,'known_prior_test_exposure':True}
    save_json(REPORTS/'experiment_identity.json',meta)
    display(pd.DataFrame([{'subset':k,'students':v} for k,v in SPLIT_PUBLIC.items()]))
    print('Data SHA',DATASET_SHA[:16],'Split SHA',SPLIT_SHA[:16], 'Difficulty SHA', DIFFICULTY_SHA[:16])

# 3. Deep Knowledge Tracing (DKT)

## 3.1 Definisi model dan prosedur pelatihan

DKT GRU memproses `(soal, respons)` secara temporal dan memperkirakan peluang jawaban benar pada soal berikutnya. Target yang digunakan untuk memberi reward kebijakan di data training **tidak** diberikan kepada DKT dalam training encoder. State DKT diperbarui hanya setelah respons replay dari soal terpilih tersedia.

- `MODE='smoke'`: DKT dilatih **dari awal** selama dua epoch; checkpoint disimpan di `dkt/smoke/`.
- `MODE='full'`: DKT dilatih **dari awal** selama 15 epoch; checkpoint disimpan di `dkt/full/`.
- `multiseed`, `ablation`, `freeze`, `final`: hanya memakai checkpoint `dkt/full/`; tidak melatih ulang DKT.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class SharedDKT(nn.Module):
        def __init__(self, qn,kn,qkc,hidden=48,emb=32):
            super().__init__()
            self.qn,self.hidden = qn,hidden
            self.token_embedding = nn.Embedding(2*qn+1,emb,padding_idx=0)
            self.gru = nn.GRU(emb,hidden,batch_first=True)
            self.question_embedding = nn.Embedding(qn,hidden)
            self.kc_proj = nn.Linear(kn,hidden,bias=False)
            self.head = nn.Sequential(nn.Linear(3*hidden,hidden),nn.ReLU(),nn.Linear(hidden,1))
            self.register_buffer('qkc',torch.as_tensor(qkc,dtype=torch.float32))
        def token(self,q,y): return 1 + q.long() + self.qn*y.long()
        def predict_logits(self,h,q):
            # q shape [N] or [B,T]; h shape [N,H] or [B,T,H].
            item = self.question_embedding(q) + self.kc_proj(self.qkc[q])
            return self.head(torch.cat([h,item,h*item],dim=-1)).squeeze(-1)
        def forward(self,tokens,questions):
            # tokens[:,0] adalah padding penanda kondisi sebelum soal pertama.
            # h_t setelah t-1 respons; identik dengan zero_state() dan step() saat CAT.
            if tokens.shape[1] > 1:
                h_after,_ = self.gru(self.token_embedding(tokens[:,1:]))
                h = torch.cat([torch.zeros_like(h_after[:,:1]),h_after],dim=1)
            else:
                h = torch.zeros(tokens.shape[0],1,self.hidden,device=tokens.device)
            return self.predict_logits(h,questions)
        @torch.no_grad()
        def step(self,h,q,y):
            qt=torch.tensor([q],dtype=torch.long,device=h.device)
            yt=torch.tensor([y],dtype=torch.long,device=h.device)
            _,next_h=self.gru(self.token_embedding(self.token(qt,yt)).unsqueeze(1),h.unsqueeze(0))
            return next_h.squeeze(0)
        @torch.no_grad()
        def probs(self,h,qs):
            q = torch.as_tensor(qs,dtype=torch.long,device=h.device)
            hh = h.expand(len(q),-1)
            return torch.sigmoid(self.predict_logits(hh,q)).detach().cpu().numpy()
        def zero_state(self): return torch.zeros(1,self.hidden,device=self.qkc.device)

    DKT = SharedDKT(Q,K,qkc,C.hidden,C.emb).to(DEVICE)

    def batches_for_dkt(sids,batch_size=32,shuffle=False,seed=42):
        rr=np.random.default_rng(seed)
        ss=list(sids)
        if shuffle: rr.shuffle(ss)
        for lo in range(0,len(ss),batch_size):
            sub=ss[lo:lo+batch_size]
            L=max(len(users[sid]) for sid in sub)
            tokens=np.zeros((len(sub),L),dtype=np.int64)
            qs=np.zeros((len(sub),L),dtype=np.int64)
            yy=np.zeros((len(sub),L),dtype=np.float32)
            mask=np.zeros((len(sub),L),dtype=np.float32)
            for i,sid in enumerate(sub):
                series=CASE_LOOKUP[sid]['active']  # target withheld from ALL DKT training/validation passes
                qq=np.array([q for q,_ in series]); lab=np.array([y for _,y in series])
                l=len(series)
                qs[i,:l]=qq; yy[i,:l]=lab;mask[i,:l]=1
                if l>1: tokens[i,1:l]=1+qq[:-1]+Q*lab[:-1]
            yield (torch.as_tensor(tokens,device=DEVICE),torch.as_tensor(qs,device=DEVICE),
                   torch.as_tensor(yy,device=DEVICE),torch.as_tensor(mask,device=DEVICE))

    def dkt_dataset_bce(sids):
        DKT.eval(); total, num=0.0,0
        with torch.no_grad():
            for t,q,y,m in batches_for_dkt(sids):
                loss=F.binary_cross_entropy_with_logits(DKT(t,q),y,reduction='none')
                total+=float((loss*m).sum());num+=int(m.sum())
        return total/max(num,1)

    # Pakai validation saja untuk pemilihan checkpoint DKT.

## 3.2 Training DKT, pemilihan checkpoint, dan resume

Checkpoint terbaik dipilih dari **BCE DKT validation**, tidak dari final test. Setelah runtime terputus, checkpoint progres memuat model dan optimizer agar training dapat diteruskan. Seed dikendalikan per epoch.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    DKT_BEST = DKT_DIR/'dkt_best.pt'
    DKT_PROGRESS = DKT_DIR/'dkt_progress.pt'
    DKT_HISTORY_PATH = DKT_DIR/'dkt_history.csv'
    DKT_SIGNATURE=hashlib.sha256(json.dumps({'data':DATASET_SHA,'split':SPLIT_SHA,
        'qdiff':DIFFICULTY_SHA,'hidden':C.hidden,'emb':C.emb,'dkt_lr':C.dkt_lr,
        'epochs':C.dkt_epochs,'smoke':C.smoke},sort_keys=True).encode()).hexdigest()

    def train_or_load_dkt():
        train_mode=MODE in ('smoke','full','kc_smoke','kc_audit')
        if not train_mode:
            if not DKT_BEST.exists():
                raise FileNotFoundError(f'DKT full belum tersedia: {DKT_BEST}. Jalankan MODE="full" dahulu.')
            identity=DKT_DIR/'dkt_identity.json'
            if not identity.exists() or json.loads(identity.read_text())['signature'] != DKT_SIGNATURE:
                raise RuntimeError('DKT identity berbeda; hentikan. Periksa dataset/split/config. Jangan hapus checkpoint.')
            DKT.load_state_dict(torch.load(DKT_BEST,map_location=DEVICE,weights_only=True))
            DKT.eval()
            return pd.read_csv(DKT_HISTORY_PATH)
        seed_everything(C.seed)
        opt=torch.optim.AdamW(DKT.parameters(),lr=C.dkt_lr,weight_decay=1e-4)
        best=float('inf');start=1;log=[]
        if DKT_PROGRESS.exists():
            data=torch.load(DKT_PROGRESS,map_location=DEVICE,weights_only=True)
            if data['signature']!=DKT_SIGNATURE: raise RuntimeError('Identitas DKT berubah. Jangan timpa checkpoint.')
            DKT.load_state_dict(data['model'])
            opt.load_state_dict(data['optimizer'])
            best=float(data['best']);start=int(data['epoch'])+1;log=data['history']
            print('Resume DKT mulai epoch',start)
        train_ids=SPLIT['train'][:C.train_students] if C.train_students else SPLIT['train']
        for ep in range(start,C.dkt_epochs+1):
            seed_everything(C.seed+ep*7919)
            DKT.train();cum=0;count=0
            for t,q,y,m in batches_for_dkt(train_ids,shuffle=True,seed=C.seed+ep):
                loss=(F.binary_cross_entropy_with_logits(DKT(t,q),y,reduction='none')*m).sum()/m.sum().clamp_min(1)
                opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(DKT.parameters(),1.0)
                opt.step();cum+=float(loss.detach());count+=1
            val=dkt_dataset_bce(DKT_VAL_IDS)
            log.append({'epoch':ep,'train_batch_bce':cum/max(1,count),'val_bce':val})
            if val<best:
                best=val;save_torch(DKT_BEST,DKT.state_dict())
            save_torch(DKT_PROGRESS,{'signature':DKT_SIGNATURE,'model':DKT.state_dict(),
                'optimizer':opt.state_dict(),'history':log,'best':best,'epoch':ep})
            pd.DataFrame(log).to_csv(DKT_HISTORY_PATH,index=False)
            print(f'DKT epoch {ep}/{C.dkt_epochs} | train={cum/max(1,count):.4f} | val={val:.4f}')
        if not DKT_BEST.exists():raise RuntimeError('Checkpoint DKT terbaik tidak ditemukan')
        DKT.load_state_dict(torch.load(DKT_BEST,map_location=DEVICE,weights_only=True))
        DKT.eval()
        save_json(DKT_DIR/'dkt_identity.json',{'signature':DKT_SIGNATURE,'checkpoint_sha256':sha256(DKT_BEST),
           'data_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'config':asdict(C)})
        return pd.DataFrame(log)

    DKT_HISTORY=train_or_load_dkt()
    print('DKT checkpoint SHA:',sha256(DKT_BEST))
    display(DKT_HISTORY.tail())

## 3.3 Diagnostik DKT dan konsistensi GRU

Periksa bahwa inference bertahap setelah sejumlah respons menghasilkan state dan prediksi yang sama dengan forward GRU pada urutan tersebut. Simpan kurva train/validation BCE sebagai PNG, SVG, dan CSV.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Urutan uji sama dengan respons kronologis yang telah dipisah.
    test_seq=CASES['dkt_val'][0]['active'][:8]
    h=DKT.zero_state()
    with torch.no_grad():
        for q,y in test_seq: h=DKT.step(h,q,y)
        q_eval=test_seq[-1][0]
        p_step=float(DKT.probs(h,[q_eval])[0])
        # Manual GRU token sama dengan step berulang, bukan final-test target.
        toks=torch.tensor([1+q+Q*y for q,y in test_seq],dtype=torch.long,device=DEVICE).unsqueeze(0)
        z,_=DKT.gru(DKT.token_embedding(toks))
        p_seq=float(torch.sigmoid(DKT.predict_logits(z[:,-1,:],torch.tensor([q_eval],device=DEVICE)))[0])
    assert abs(p_step-p_seq)<1e-5,(p_step,p_seq)
    fig,ax=plt.subplots(figsize=(7.8,3.5))
    ax.plot(DKT_HISTORY.epoch,DKT_HISTORY.train_batch_bce,marker='o',label='Training BCE')
    ax.plot(DKT_HISTORY.epoch,DKT_HISTORY.val_bce,marker='s',label='Validation BCE')
    ax.set(xlabel='Epoch',ylabel='BCE',title='Kurva training DKT')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(FIGURES/'01_dkt_learning_curve.png',dpi=170)
    fig.savefig(FIGURES/'01_dkt_learning_curve.svg')
    display(fig);plt.close(fig)

# 4. CAT offline: definisi state, action, reward dan metrik

## 4.1 State dan action

- State: `h_DKT`, KC yang telah dicakup, dan fraksi panjang tes.
- Global state untuk critic: state + ketersediaan KC yang tersisa.
- Fitur soal: peluang benar DKT, entropi, kebaruan KC, dan difficulty train-only.
- Action: indeks soal dari interaksi yang pernah teramati, tanpa pengulangan.
- MARL: Planning memilih strategi, KC Agent memilih konsep layak, Item Agent memilih soal valid yang memuat konsep tersebut.

## 4.2 Reward multi-objective berbasis data training

\[r_t=1.0\,\Delta KC+1.5\,\Delta BCE+0.8\,\Delta Rank+0.07\,H(q_t)/20\]

BCE gain dan ranking gain dihitung terhadap interaksi target **mahasiswa training saja**. Untuk policy selection/audit/final, target digunakan **hanya saat menghitung metrik**, bukan reward atau pemilihan soal. `rank` adalah surrogate sigmoid untuk ranking prediksi benar versus salah; bukan AUC aktual yang dipakai sebagai reward.

## 4.3 Metrik

**KC Available** = KC yang tercakup / KC yang tersedia dalam kumpulan soal mahasiswa. **KC Global** = KC yang tercakup / seluruh KC dataset. **AUC** dihitung dengan menggabungkan prediksi jawaban target semua mahasiswa dalam kelompok evaluasi; **BCE** menghitung rata-rata loss semua target. `length` diharapkan 20 untuk seluruh metode.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class OfflineCAT:
        def __init__(self,case):
            self.sid=case['sid']
            self.active=dict(case['active'])
            self.target_q=np.array([q for q,_ in case['target']],dtype=np.int64)
            self.target_y=np.array([y for _,y in case['target']],dtype=np.float32)
            self.eligible=np.zeros(Q,dtype=bool)
            self.eligible[list(self.active)]=True
            self.available_kc = np.asarray(qkc[self.eligible].sum(axis=0)>0,dtype=bool)
            self.total_eligible=int(self.eligible.sum())
            self.reset()
        def reset(self):
            self.h=DKT.zero_state().detach()
            self.remaining=self.eligible.copy()
            self.covered=np.zeros(K,dtype=bool)
            self.n=0
            self.chosen=[]
            return self.obs()
        def obs(self):
            hh=self.h.detach().cpu().numpy().reshape(-1)
            frac=np.array([self.n/C.test_length],dtype=np.float32)
            state=np.concatenate([hh,self.covered.astype(np.float32),frac]).astype(np.float32)
            remk=(qkc[self.remaining].sum(axis=0)/max(self.total_eligible,1)).astype(np.float32)
            global_obs=np.concatenate([state,remk]).astype(np.float32)
            # Semata-mata dari h dan metadata item, bukan dari jawaban target.
            p=DKT.probs(self.h,np.arange(Q)).astype(np.float32)
            entropy=-(p*np.log(p+1e-7)+(1-p)*np.log(1-p+1e-7))/np.log(2)
            novelty=(qkc*(~self.covered)[None,:]).sum(axis=1)/np.maximum(1,qkc.sum(axis=1))
            features=np.column_stack([p,entropy,novelty,qdiff]).astype(np.float32)
            feasible_kc=np.asarray(qkc[self.remaining].sum(axis=0)>0,dtype=bool)
            # Project the fixed KC scores onto each item's linked, feasible KCs.
            # Fixed Balanced weights mirror the hierarchical prior's Balanced option.
            kc_counts = qkc[self.remaining].sum(axis=0)
            kc_uncertainty = (qkc.T @ (entropy * self.remaining)) / np.maximum(kc_counts, 1)
            kc_bias = .44 * (~self.covered) + .40 * kc_uncertainty
            item_kc_bias = (qkc @ kc_bias) / qkc.sum(axis=1)
            matched_prior = (item_kc_bias + .65 * novelty - .07 * entropy).astype(np.float32)
            return {'matched_prior':matched_prior, 'state':state,'global_obs':global_obs,'features':features,'mask_q':self.remaining.copy(),
                    'mask_k':feasible_kc,'covered':self.covered.copy()}
        def target_bce(self):
            pred=np.clip(DKT.probs(self.h,self.target_q),1e-6,1-1e-6)
            return float(np.mean(-self.target_y*np.log(pred)-(1-self.target_y)*np.log(1-pred)))
        def act(self,q,training=False):
            q=int(q)
            if q<0 or q>=Q or not self.remaining[q]:raise ValueError(f'Pilihan tidak tersedia: {q}')
            before=self.target_bce() if training else 0.0
            pq=float(DKT.probs(self.h,[q])[0])
            inf=float(-(pq*np.log(pq+1e-7)+(1-pq)*np.log(1-pq+1e-7))/np.log(2))
            new=np.asarray((qkc[q]>0)&(~self.covered),dtype=bool)
            coverage_gain=float(new.sum()) / max(1,int(self.available_kc.sum()))
            y=int(self.active[q]);self.h=DKT.step(self.h,q,y)
            self.covered |= (qkc[q]>0)
            self.remaining[q]=False;self.chosen.append(q);self.n+=1
            done=self.n>=C.test_length
            if training:
                bce_gain=np.clip(before-self.target_bce(),-0.3,0.3)
                reward=C.reward_coverage*coverage_gain + C.reward_bce_gain*bce_gain + C.reward_information*inf/C.test_length
            else:
                reward=0.0
            return self.obs(),float(reward),done
        def result(self):
            pred=np.clip(DKT.probs(self.h,self.target_q),1e-6,1-1e-6)
            auc=roc_auc_score(self.target_y,pred) if len(set(self.target_y.tolist()))==2 else np.nan
            return {'sid':self.sid,'kc_global':float(self.covered.mean()),
                    'kc_available':float(self.covered.sum()/max(1,self.available_kc.sum())),
                    'bce':float(np.mean(-self.target_y*np.log(pred)-(1-self.target_y)*np.log(1-pred))),
                    'auc_student':float(auc),'length':self.n,'y':self.target_y.copy(),'p':pred.copy()}

    example=OfflineCAT(CASES['train'][0]);o=example.reset()
    assert o['mask_q'].sum()>=C.test_length
    assert not np.any(np.isin(example.target_q,np.where(o['mask_q'])[0]))
    print('State:',o['state'].shape,'Global:',o['global_obs'].shape,'Features:',o['features'].shape)
    OldOfflineCAT=OfflineCAT
    from scipy.special import expit
    class OfflineCAT(OldOfflineCAT):
     def rank_surrogate(self):
      p=DKT.probs(self.h,self.target_q)
      pos=p[self.target_y==1];neg=p[self.target_y==0]
      if len(pos)==0 or len(neg)==0:return 0.0
      return float(expit((pos[:,None]-neg[None,:])/0.15).mean())
     def act(self,q,training=False):
      if not training:return super().act(q,False)
      prev=self.target_bce(); rank_before=self.rank_surrogate()
      oldcov=C.reward_coverage;oldbce=C.reward_bce_gain;oldinf=C.reward_information
      # base routine supplies consistent action+state transition and coverage/entropy gain
      obs,rew,done=super().act(q,True)
      rank_gain=float(np.clip(self.rank_surrogate()-rank_before,-0.20,0.20))
      return obs,float(rew+C.reward_rank*rank_gain),done

## 4.4 Pemeriksaan lingkungan

Uji ketersediaan aksi, larangan target dalam kandidat soal, reward finite, dan panjang tes. Uji ini **tidak** memakai mahasiswa final test.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    probe=OfflineCAT(CASES['train'][0]);obs=probe.reset()
    assert obs['state'].shape==(C.hidden+K+1,)
    assert obs['features'].shape==(Q,4)
    assert obs['mask_q'].sum()>=C.test_length
    assert not(set(probe.target_q)&set(np.flatnonzero(obs['mask_q'])))
    first=int(np.flatnonzero(obs['mask_q'])[0]);_,reward,_=probe.act(first,training=True)
    assert np.isfinite(reward) and probe.n==1 and not probe.remaining[first]
    print('CAT unit test berhasil; reward training pertama:',round(reward,5))

# 5. Dua baseline dan tiga agen MARL

## 5.1 Definisi arsitektur

**RL-Flat:** logits langsung untuk seluruh soal dari state DKT. **RL-Feat:** logits untuk seluruh soal dari state dan fitur soal. **MARL:** Planning Agent → KC Selection Agent → Item Selection Agent. Ketiga aktor MARL dioptimasi bersama dengan cooperative PPO dan satu critic bersama. Ketiga kebijakan menggunakan **shared action prior** yang sama; ablation dapat menonaktifkannya secara eksplisit.

Desain kebijakan diagnostik digunakan sebagai dasar, tetapi **tidak ada nama varian eksperimen dalam objek model atau nama output**. Nama model utama cukup `marl`.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    def tofloat(x):return torch.as_tensor(x,dtype=torch.float32,device=DEVICE)
    def mask_dist(logits,mask):
        m=torch.as_tensor(mask,dtype=torch.bool,device=DEVICE)
        if not torch.any(m):raise ValueError('Semua aksi terlarang')
        return torch.distributions.Categorical(logits=logits.masked_fill(~m,-1e9))

    class FlatActor(nn.Module):
        def __init__(self,dim):
            super().__init__()
            self.net=nn.Sequential(nn.Linear(dim,96),nn.Tanh(),nn.Linear(96,Q))
        def forward(self,state,feat):return self.net(state)

    class FeatActor(nn.Module):
        def __init__(self,dim):
            super().__init__()
            self.state=nn.Sequential(nn.Linear(dim,64),nn.Tanh())
            self.feature=nn.Linear(4,64)
            self.item=nn.Embedding(Q,64)
            self.score=nn.Sequential(nn.Tanh(),nn.Linear(64,1))
        def forward(self,state,features):
            x=self.state(state).unsqueeze(0)+self.feature(features)+self.item.weight
            return self.score(x).squeeze(-1)


    class PlanKCItemActor(nn.Module):
     def __init__(self,dim):
      super().__init__();
      self.plan=nn.Sequential(nn.Linear(dim+4,80),nn.Tanh(),nn.Linear(80,3))
      self.kc=nn.Sequential(nn.Linear(dim+3+4,96),nn.Tanh(),nn.Linear(96,K))
      self.state_proj=nn.Linear(dim,64)
      self.feature_proj=nn.Linear(7,64)
      self.item_embeddings=nn.Embedding(Q,64)
      self.plan_embeddings=nn.Embedding(3,64)
      self.kc_embeddings=nn.Embedding(K,64)
      self.score=nn.Sequential(nn.Tanh(),nn.Linear(64,1))
     def summary(self,obs):
      f=tofloat(obs['features']);mask=tofloat(obs['mask_q'].astype(np.float32))
      p=f[:,0]; ent=f[:,1]
      return torch.stack([f[:,2].mul(mask).sum()/mask.sum().clamp_min(1),
       ent.mul(mask).sum()/mask.sum().clamp_min(1),
       tofloat(obs['covered'].astype(float)).mean(),
       mask.mean()])
     def plan_logits(self,obs):
      x=torch.cat([tofloat(obs['state']),self.summary(obs)])
      return self.plan(x)
     def kc_logits(self,obs,plan):
      s=tofloat(obs['state']);f=tofloat(obs['features']);mask=tofloat(obs['mask_q'].astype(np.float32))
      ohp=F.one_hot(torch.tensor(plan,device=DEVICE),3).float()
      x=torch.cat([s,ohp,self.summary(obs)])
      logits=self.kc(x)
      kc_count=(tofloat(qkc).T@mask).clamp_min(1)
      uncertainty=(tofloat(qkc).T @ (f[:,1]*mask))/kc_count
      ncovered=1-tofloat(obs['covered'].astype(np.float32))
      strength=1.0
      return logits if getattr(self,'no_kc_prior',False) else logits+strength*(0.85*ncovered if plan==0 else (0.85*uncertainty if plan==1 else 0.44*ncovered+0.40*uncertainty))
     def item_logits(self,obs,plan,kc):
      st=tofloat(obs['state']);feats=tofloat(obs['features']);
      match=tofloat(qkc[:,kc]);n_new=feats[:,2]
      # 7 per-item diagnostics = probability, entropy, novelty, difficulty,
      # KC match, uncertainty*match, novelty*match.
      f=torch.cat([feats,match[:,None],(feats[:,1]*match)[:,None],(n_new*match)[:,None]],dim=1)
      x=(self.state_proj(st)[None,:] + self.feature_proj(f) + self.item_embeddings.weight
          + self.plan_embeddings(torch.tensor(plan,device=DEVICE))[None,:]
          + self.kc_embeddings(torch.tensor(kc,device=DEVICE))[None,:])
      logits=self.score(x).squeeze(-1)
      # fixed semantic priors, never depend on target answer
      if not getattr(self,'no_semantic_prior',False):
       logits=logits + 0.65*f[:,6] + 0.18*f[:,5] - 0.25*feats[:,1]
      return logits

## 5.2 Joint policy dan critic

Log-probabilitas MARL merupakan jumlah log-prob dari ketiga pemilihan: `log π(plan|state) + log π(kc|state,plan) + log π(item|state,plan,kc)`. Aksi tidak valid di-*mask* sebelum sampling. `no_planning` menggunakan strategi Balanced tetap, bukan menghapus KC/Item Agent.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class Policy(nn.Module):
        def __init__(self,kind,ablation='none'):
            super().__init__()
            assert kind in ('rl_flat','rl_feat','marl','rl_kc')
            self.kind=kind;self.ablation=ablation
            self.use_action_prior=bool(C.shared_prior and ablation!='no_prior')
            dim=C.hidden+K+1
            if kind=='rl_flat':self.actor=FlatActor(dim)
            elif kind in ('rl_feat','rl_kc'):self.actor=FeatActor(dim)
            else:
                self.actor=PlanKCItemActor(dim)
                self.actor.no_semantic_prior=ablation in ('no_semantic_prior','no_special_priors')
                self.actor.no_kc_prior=ablation=='no_special_priors'
            self.critic=nn.Sequential(nn.Linear(dim+K,96),nn.Tanh(),nn.Linear(96,1))
        def value(self,obs):return self.critic(tofloat(obs['global_obs'])).squeeze(-1)
        def choice(self,obs,greedy=False,actions=None):
            def pick(dist,forced=None):
                a=(int(dist.probs.argmax()) if greedy else int(dist.sample())) if forced is None else int(forced)
                return a,dist.log_prob(torch.tensor(a,device=DEVICE)),dist.entropy()
            if self.kind!='marl':
                logits=self.actor(tofloat(obs['state']),tofloat(obs['features']))
                if self.use_action_prior:
                    logits=logits+C.shared_prior_entropy*tofloat(obs['features'])[:,1]+C.shared_prior_difficulty*tofloat(obs['features'])[:,3]
                if self.kind=='rl_kc':
                    logits=logits+tofloat(obs['matched_prior'])
                dist=mask_dist(logits,obs['mask_q'])
                q,lp,ent=pick(dist,None if actions is None else actions['q'])
                return {'q':q},lp,ent
            if self.ablation=='no_planning':
                plan,l0,e0=2,torch.tensor(0.,device=DEVICE),torch.tensor(0.,device=DEVICE)
            else:
                d0=torch.distributions.Categorical(logits=self.actor.plan_logits(obs))
                plan,l0,e0=pick(d0,None if actions is None else actions['plan'])
            kd=mask_dist(self.actor.kc_logits(obs,plan),obs['mask_k'])
            kc,l1,e1=pick(kd,None if actions is None else actions['kc'])
            qlogits=self.actor.item_logits(obs,plan,kc)
            if self.use_action_prior:
                qlogits=qlogits+C.shared_prior_entropy*tofloat(obs['features'])[:,1]+C.shared_prior_difficulty*tofloat(obs['features'])[:,3]
            qm=obs['mask_q']&(qkc[:,kc]>0)
            qd=mask_dist(qlogits,qm)
            q,l2,e2=pick(qd,None if actions is None else actions['q'])
            return {'plan':plan,'kc':kc,'q':q},l0+l1+l2,e0+e1+e2

## 5.3 Unit test masking, shared DKT, dan gradien

Cek bahwa ketiga agen menghasilkan aksi valid, DKT tetap `frozen` saat training PPO, dan jalur gradien model berbeda sesuai baseline. **Catatan adil:** RL-Flat tidak mempunyai fitur item conditioning, sementara RL-Feat mempunyai fitur. Perbedaan kapasitas parameter dilaporkan secara terbuka.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    obs=OfflineCAT(CASES['train'][1]).reset()
    for name in ('rl_flat','rl_feat','marl','rl_kc'):
        p=Policy(name).to(DEVICE)
        act,lp,ent=p.choice(obs)
        assert obs['mask_q'][act['q']] and torch.isfinite(lp) and torch.isfinite(ent)
        if name=='marl':assert qkc[act['q'],act['kc']]>0
        print(name,'parameters',sum(t.numel() for t in p.parameters()),'example action',act)
    assert not any(x.requires_grad is False for x in Policy('marl').parameters())
    for param in DKT.parameters():param.requires_grad_(False)
    DKT.eval()

# 6. Training PPO, validasi, checkpoint, dan resume

## 6.1 Rollout dan evaluasi

Episode memakai respons teramati (offline replay), tetapi posisi soal **dipilih** oleh kebijakan. Reward diagnosis hanya dihitung pada mahasiswa training. Metrik policy selection dipakai untuk memilih checkpoint; policy audit dipakai untuk membaca generalisasi terhadap mahasiswa validation lain. Semua seed mengikuti anggaran training yang sama.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    def get_cases(split):
        cases=CASES['policy_audit'] if split=='audit' else CASES[split]
        if split=='test' and MODE!='final': raise RuntimeError('Final test terkunci. Akses hanya di MODE=final.')
        if split=='train' and C.train_students: return cases[:C.train_students]
        if split=='val':return CASES['policy_select']
        if split=='audit':return CASES['policy_audit']
        return cases

    @torch.no_grad()
    def evaluate(policy,cases,greedy=True):
        policy.eval();rows=[]
        for case in cases:
            env=OfflineCAT(case);obs=env.reset()
            for t in range(C.test_length):
                acts,_,_=policy.choice(obs,greedy=greedy)
                obs,_,done=env.act(acts['q'],training=False)
                if done:break
            if MODE in ('kc_smoke','kc_audit'):
                assert env.n == C.test_length and len(set(env.chosen)) == C.test_length
                assert all(env.eligible[q] for q in env.chosen)
            rows.append(env.result())
        return rows

    def summarize(rows):
        y=np.concatenate([r['y'] for r in rows]);p=np.concatenate([r['p'] for r in rows])
        return {'kc_available':float(np.mean([r['kc_available'] for r in rows])),
                'kc_global':float(np.mean([r['kc_global'] for r in rows])),
                'auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else float('nan'),
                'bce':float(np.mean(-y*np.log(p)-(1-y)*np.log(1-p))),
                'length':float(np.mean([r['length'] for r in rows]))}

    def val_utility(metrics):
        # Tetapkan sebelum pelatihan; hanya val, tanpa akses test.
        # Skala utilitas ini dipakai untuk checkpoint selection, bukan untuk mendefinisikan kemenangan.
        return metrics['kc_available'] + 0.5*metrics['auc'] - 0.5*metrics['bce']

    def run_episode(policy,case):
        env=OfflineCAT(case);obs=env.reset();transition=[]
        for t in range(C.test_length):
            with torch.no_grad():
                actions,logp,_=policy.choice(obs)
                v=float(policy.value(obs))
            next_obs,reward,done=env.act(actions['q'],training=True)
            transition.append({'obs':obs,'actions':actions,'logp_old':float(logp),'old_v':v,
                               'reward':reward,'done':done})
            obs=next_obs
            if done:break
        # Monte-Carlo episode sudah selesai; advantage GAE.
        gae=0.0;next_value=0.0
        for r in reversed(transition):
            delta=r['reward']+C.gamma*next_value*(not r['done'])-r['old_v']
            gae=delta+C.gamma*C.gae_lambda*(not r['done'])*gae
            r['adv']=gae;r['return']=gae+r['old_v'];next_value=r['old_v']
        return transition

    def ppo_update(policy,optimizer,data):
        # Intentionally simple per-trajectory loops for interpretability; can be batched later.
        advantages=np.asarray([r['adv'] for r in data],dtype=np.float32)
        advantages=(advantages-advantages.mean())/(advantages.std()+1e-6)
        ids=np.arange(len(data)); losses=[]
        for ep in range(C.ppo_epochs):
            np.random.shuffle(ids)
            for chunk in np.array_split(ids,max(1,int(np.ceil(len(ids)/48)))):
                pl=[];vl=[];en=[]
                for ix in chunk:
                    r=data[ix]
                    _,new_lp,entropy=policy.choice(r['obs'],actions=r['actions'])
                    ratio=torch.exp(new_lp-torch.tensor(r['logp_old'],device=DEVICE))
                    adv=tofloat(advantages[ix]);s1=ratio*adv;s2=torch.clamp(ratio,1-C.clip,1+C.clip)*adv
                    pl.append(-torch.minimum(s1,s2))
                    vl.append(F.mse_loss(policy.value(r['obs']),tofloat(r['return'])))
                    en.append(entropy)
                loss=torch.stack(pl).mean()+0.5*torch.stack(vl).mean()-0.001*torch.stack(en).mean()
                optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(policy.parameters(),0.5)
                optimizer.step();losses.append(float(loss.detach()))
        return float(np.mean(losses))

## 6.2 PPO vektor dan kesetaraan log-probabilitas

PPO penuh menggunakan mini-batch berbasis vektor agar lebih cepat daripada loop per-item. Rumus clipped PPO sama dengan implementasi smoke. Uji di bawah memeriksa log-prob hasil batch terhadap log-prob `choice(..., actions=...)` dan menghindari perbedaan aturan mask.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Batching PPO untuk training penuh: matematika clipped PPO sama,
    # tetapi forward policy untuk seluruh mini-batch menggantikan loop satu aksi.
    def vectorized_ppo_update(policy, optimizer, data):
        advantage=np.asarray([r['adv'] for r in data],dtype=np.float32)
        advantage=(advantage-advantage.mean())/(advantage.std()+1e-6)
        ids=np.arange(len(data)); losses=[]
        qkc_t=tofloat(qkc)
        for epoch in range(C.ppo_epochs):
            np.random.shuffle(ids)
            for ind in np.array_split(ids,max(1,int(np.ceil(len(ids)/48)))):
                rr=[data[int(j)] for j in ind]
                st=tofloat(np.stack([r['obs']['state'] for r in rr]))
                gl=tofloat(np.stack([r['obs']['global_obs'] for r in rr]))
                ft=tofloat(np.stack([r['obs']['features'] for r in rr]))
                qm=torch.as_tensor(np.stack([r['obs']['mask_q'] for r in rr]),device=DEVICE,dtype=torch.bool)
                km=torch.as_tensor(np.stack([r['obs']['mask_k'] for r in rr]),device=DEVICE,dtype=torch.bool)
                av=tofloat(advantage[ind]);oldlp=tofloat([r['logp_old'] for r in rr]); ret=tofloat([r['return'] for r in rr])
                selected_q=torch.tensor([r['actions']['q'] for r in rr],device=DEVICE,dtype=torch.long)
                def dist_mask(logits,mask):
                    return torch.distributions.Categorical(logits=logits.masked_fill(~mask,-1e9))
                if policy.kind=='rl_flat':
                    lg=policy.actor.net(st)
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d=dist_mask(lg,qm);lp=d.log_prob(selected_q);entropy=d.entropy()
                elif policy.kind in ('rl_feat','rl_kc'):
                    x=(policy.actor.state(st)[:,None,:]
                       + policy.actor.feature(ft)+policy.actor.item.weight[None,:,:])
                    lg=policy.actor.score(x).squeeze(-1)
                    if policy.kind=='rl_kc':
                        lg=lg+tofloat(np.stack([r['obs']['matched_prior'] for r in rr]))
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d=dist_mask(lg,qm);lp=d.log_prob(selected_q);entropy=d.entropy()
                else:
                    a=policy.actor
                    selected_plan=torch.tensor([r['actions']['plan'] for r in rr],device=DEVICE,dtype=torch.long)
                    selected_kc=torch.tensor([r['actions']['kc'] for r in rr],device=DEVICE,dtype=torch.long)
                    mask_f=qm.float()
                    cov=tofloat(np.stack([r['obs']['covered'] for r in rr]).astype(np.float32))
                    den=mask_f.sum(1).clamp_min(1)
                    summary=torch.stack([(ft[:,:,2]*mask_f).sum(1)/den,
                        (ft[:,:,1]*mask_f).sum(1)/den,cov.mean(1),mask_f.mean(1)],dim=1)
                    if policy.ablation=='no_planning':
                        log0=torch.zeros(len(rr),device=DEVICE);ent0=log0
                    else:
                        d0=torch.distributions.Categorical(logits=a.plan(torch.cat([st,summary],dim=1)))
                        log0=d0.log_prob(selected_plan);ent0=d0.entropy()
                    ohp=F.one_hot(selected_plan,3).float()
                    kc_logit=a.kc(torch.cat([st,ohp,summary],dim=1))
                    kc_unc=((ft[:,:,1]*mask_f)@qkc_t)/((mask_f@qkc_t).clamp_min(1))
                    novelty=1-cov
                    strength=1.0
                    wnew=torch.where(selected_plan==0,.85,torch.where(selected_plan==1,0.0,.44))
                    wunc=torch.where(selected_plan==0,0.0,torch.where(selected_plan==1,.85,.40))
                    if not getattr(a,'no_kc_prior',False):
                        kc_logit=kc_logit+strength*(wnew[:,None]*novelty+wunc[:,None]*kc_unc)
                    d1=dist_mask(kc_logit,km);log1=d1.log_prob(selected_kc);ent1=d1.entropy()
                    match=qkc_t[:,selected_kc].T
                    features=torch.cat([ft,match[:,:,None],(ft[:,:,1]*match)[:,:,None],(ft[:,:,2]*match)[:,:,None]],dim=2)
                    xx=(a.state_proj(st)[:,None,:]+a.feature_proj(features)
                        +a.item_embeddings.weight[None,:,:]
                        +a.plan_embeddings(selected_plan)[:,None,:]
                        +a.kc_embeddings(selected_kc)[:,None,:])
                    lg=a.score(xx).squeeze(-1)
                    if not getattr(a,'no_semantic_prior',False):
                        lg=lg+.65*features[:,:,6]+.18*features[:,:,5]-.25*ft[:,:,1]
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d2=dist_mask(lg,qm&(match>0))
                    lp=log0+log1+d2.log_prob(selected_q)
                    entropy=ent0+ent1+d2.entropy()
                ratio=torch.exp(lp-oldlp)
                loss_policy=-torch.minimum(ratio*av,torch.clamp(ratio,1-C.clip,1+C.clip)*av).mean()
                value=policy.critic(gl).squeeze(-1)
                loss=loss_policy+.5*F.mse_loss(value,ret)-.001*entropy.mean()
                optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(policy.parameters(),.5)
                optimizer.step();losses.append(float(loss.detach()))
        return float(np.mean(losses))

## 6.3 Resume aman setiap update

File PPO progres disimpan setelah setiap update. Ketika Colab disconnect, notebook membaca optimizer, model, histori, dan nomor update. SHA dataset, split, DKT, serta konfigurasi dicocokkan sebelum melanjutkan.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    POLICIES={};HISTORIES={}

    def train_policy(kind,seed=42,ablation='none'):
        if MODE not in ('smoke','full','multiseed','ablation','kc_smoke','kc_audit'):
            raise RuntimeError('Training tidak boleh dijalankan saat freeze/final')
        seed_everything(seed)
        old_cov,old_info,old_rank=C.reward_coverage,C.reward_information,C.reward_rank
        if ablation=='no_coverage_reward':C.reward_coverage=0.0
        if ablation=='no_information_reward':C.reward_information=0.0
        if ablation=='no_rank_reward':C.reward_rank=0.0
        try:
            model=Policy(kind,ablation).to(DEVICE)
            optimizer=torch.optim.Adam(model.parameters(),lr=C.ppo_lr)
            label=kind if ablation=='none' else f'{kind}_{ablation}'
            run_dir=MODE_ROOT/f'seed{seed}'
            (run_dir/'ppo').mkdir(parents=True,exist_ok=True)
            ckptdir=run_dir/'ppo'
            progress=ckptdir/f'{label}_progress.pt'
            best_path=ckptdir/f'{label}_best.pt'
            history_path=run_dir/f'{label}_history.csv'
            training_id={ 'mode':MODE,'kind':kind,'ablation':ablation,'seed':seed,
              'data_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'dkt_sha256':sha256(DKT_BEST),
              'difficulty_sha256':DIFFICULTY_SHA,'config':asdict(C)}
            training_sha=hashlib.sha256(json.dumps(training_id,sort_keys=True).encode()).hexdigest()
            best=-float('inf');start=1;log=[]
            if progress.exists():
                cp=torch.load(progress,map_location=DEVICE,weights_only=True)
                if cp['signature']!=training_sha:
                    raise RuntimeError(f'Checkpoint {label} tidak cocok identitas eksperimen. Jangan timpa file.')
                model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
                log=cp['history'];best=float(cp['best']);start=int(cp['update'])+1
                print(f'RESUME {label} seed={seed} dari update {start}')
            for upd in range(start,C.updates+1):
                seed_everything(seed+upd*1237)
                rr=np.random.default_rng(seed+upd*1139)
                model.train();transitions=[]
                cases=get_cases('train')
                sampled=rr.choice(len(cases),size=C.episodes_update,replace=len(cases)<C.episodes_update)
                for idx in sampled:transitions.extend(run_episode(model,cases[int(idx)]))
                loss=(ppo_update(model,optimizer,transitions) if C.smoke
                      else vectorized_ppo_update(model,optimizer,transitions))
                validate=C.smoke or upd==1 or upd%C.validation_interval_full==0 or upd==C.updates
                if validate:
                    score=summarize(evaluate(model,get_cases('val')))
                    u=val_utility(score)
                    log.append({'update':upd,'loss':loss,'utility_val':u,**score})
                    if u>best:
                        best=u;save_torch(best_path,model.state_dict())
                    print(f'{label} seed{seed} update {upd}/{C.updates} KC={score["kc_available"]:.4f} AUC={score["auc"]:.4f} BCE={score["bce"]:.4f} utility={u:.4f}',flush=True)
                else: log.append({'update':upd,'loss':loss,'utility_val':float('nan')})
                save_torch(progress,{'signature':training_sha,'model':model.state_dict(),
                    'optimizer':optimizer.state_dict(),'history':log,'best':best,'update':upd})
                pd.DataFrame(log).to_csv(history_path,index=False)
            if not best_path.exists():raise RuntimeError('Checkpoint policy terbaik tidak ada: '+str(best_path))
            model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=True))
            model.eval()
            return model,pd.DataFrame(log)
        finally:
            C.reward_coverage=old_cov;C.reward_information=old_info;C.reward_rank=old_rank

# 7. Eksperimen training dan validasi

## 7.1 Training RL-Flat, RL-Feat, dan MARL

- `smoke`: 2 update × 2 episode × 3 kebijakan; DKT dilatih 2 epoch dari awal.
- `full`: 50 update × 20 episode × 3 kebijakan; DKT dilatih 15 epoch dari awal.
- `multiseed`: DKT checkpoint full yang sama, tetapi policy PPO diulang pada lima seed.
- `ablation`: DKT full yang sama; MARL dan variasi ablasinya dilatih ulang untuk dua seed.

**Tidak ada test set yang dievaluasi pada tahap ini.**

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    SEEDS = (MULTI_SEEDS if MODE=='multiseed' else ABLATION_SEEDS if MODE=='ablation' else [SEED])
    MODELS=('rl_flat','rl_feat','marl')
    ABLATIONS=('none','no_planning','no_rank_reward','no_coverage_reward','no_semantic_prior')
    RUN_RESULTS=[];ALL_HISTORY={};MODELS_BY_SEED={};AUDIT_ROWS={}
    if MODE in ('smoke','full','multiseed','ablation'):
        for seed in SEEDS:
            MODELS_BY_SEED[seed]={}
            jobs=[('marl',a) for a in ABLATIONS] if MODE=='ablation' else [(name,'none') for name in MODELS]
            for kind,abl in jobs:
                label=kind if abl=='none' else f'{kind}_{abl}'
                print('\nTRAINING',label,'seed',seed,flush=True)
                model,hist=train_policy(kind,seed,abl)
                MODELS_BY_SEED[seed][label]=model
                ALL_HISTORY[(seed,label)]=hist
                select_rows=evaluate(model,get_cases('val'))
                audit_rows=evaluate(model,get_cases('audit')[:20] if MODE=='smoke' else get_cases('audit'))
                AUDIT_ROWS[(seed,label)]=audit_rows
                for subset,rows in [('selection',select_rows),('audit20' if MODE=='smoke' else 'audit',audit_rows)]:
                    RUN_RESULTS.append({'seed':seed,'model':label,'subset':subset,**summarize(rows)})
                    pd.DataFrame(RUN_RESULTS).to_csv(TABLES/'results_progress.csv',index=False)
        RESULTS=pd.DataFrame(RUN_RESULTS)
        RESULTS.to_csv(TABLES/'model_comparison.csv',index=False)
        display(RESULTS.round(5))
    else:
        RESULTS=pd.DataFrame()
        print('MODE',MODE,'tidak melatih kebijakan; model frozen akan dimuat dari full.')

## 7.2 Audit keputusan dan pengujian kemenangan tiga metrik

Perbandingan harus menggunakan **baseline terbaik untuk masing-masing metrik**. Perubahan hasil pada subset yang sudah dilihat tidak boleh disebut keberhasilan konfirmatori. Semua tabel dan checkpoint setiap seed disimpan terpisah.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    if not RESULTS.empty and MODE!='ablation':
        report=[]
        for (seed,subset),frame in RESULTS.groupby(['seed','subset']):
            rows=frame.set_index('model')
            win_kc=rows.loc['marl','kc_available']>rows.loc[['rl_flat','rl_feat'],'kc_available'].max()
            win_auc=rows.loc['marl','auc']>rows.loc[['rl_flat','rl_feat'],'auc'].max()
            win_bce=rows.loc['marl','bce']<rows.loc[['rl_flat','rl_feat'],'bce'].min()
            report.append({'seed':seed,'subset':subset,'kc_win':win_kc,'auc_win':win_auc,
                           'bce_win':win_bce,'all_three':bool(win_kc and win_auc and win_bce)})
        WIN_REPORT=pd.DataFrame(report)
        WIN_REPORT.to_csv(TABLES/'win_report.csv',index=False)
        display(WIN_REPORT)
    else:
        WIN_REPORT=pd.DataFrame()

# 8. Tabel dan visualisasi penelitian

## 8.1 Grafik perbandingan dan variasi antar-seed

Visualisasi akan ditampilkan di notebook sekaligus disimpan dalam PNG/SVG. File CSV disimpan di `tables/`, riwayat update di folder `runs/<mode>/seed<seed>/`.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    if not RESULTS.empty:
        for subset,part in RESULTS.groupby('subset'):
            means=part.groupby('model')[['kc_available','auc','bce']].mean()
            sds=part.groupby('model')[['kc_available','auc','bce']].std().fillna(0)
            means.to_csv(TABLES/f'{subset}_mean.csv')
            sds.to_csv(TABLES/f'{subset}_std.csv')
            fig,axes=plt.subplots(1,3,figsize=(13.5,3.7))
            for ax,m,title in zip(axes,['kc_available','auc','bce'],['KC Available ↑','AUC ↑','BCE ↓']):
                ax.bar(means.index,means[m],yerr=sds[m] if len(part.seed.unique())>1 else None,capsize=3)
                ax.set_title(f'{title} — {subset}');ax.tick_params(axis='x',labelrotation=25)
                ax.grid(axis='y',alpha=.2)
            fig.tight_layout()
            fig.savefig(FIGURES/f'02_{subset}_comparison.png',dpi=170)
            fig.savefig(FIGURES/f'02_{subset}_comparison.svg')
            display(fig);plt.close(fig)
        for seed in SEEDS:
            fig,axes=plt.subplots(1,3,figsize=(13.5,3.5))
            for label in MODELS_BY_SEED[seed]:
                h=ALL_HISTORY[(seed,label)]
                valid=h[h.utility_val.notna()]
                for ax,m in zip(axes,['kc_available','auc','bce']):
                    if m in valid.columns:ax.plot(valid['update'],valid[m],marker='.',label=label)
            for ax,title in zip(axes,['KC Available ↑','AUC ↑','BCE ↓']):
                ax.set_title(title);ax.grid(alpha=.2);ax.set_xlabel('PPO update')
            axes[0].legend(fontsize=7);fig.tight_layout()
            fig.savefig(FIGURES/f'03_ppo_seed{seed}.png',dpi=170)
            fig.savefig(FIGURES/f'03_ppo_seed{seed}.svg')
            display(fig);plt.close(fig)
    else:
        print('Grafik training dilewati pada mode',MODE)

## 8.2 Perbandingan agregat multi-seed

Lima seed mencerminkan variasi training PPO. Laporkan mean ± SD **dan** jumlah seed dengan kemenangan di semua metrik. Jangan menyamakan mean terbaik dengan kemenangan pada setiap seed.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    if MODE=='multiseed':
        for subset,part in RESULTS.groupby('subset'):
            agg=part.groupby('model')[['kc_available','kc_global','auc','bce','length']].agg(['mean','std'])
            display(agg.round(5))
            agg.to_csv(TABLES/f'{subset}_multiseed_mean_std.csv')
            print('Kemenangan MARL semua metrik:',int(WIN_REPORT[WIN_REPORT.subset==subset].all_three.sum()),'/',len(SEEDS),'seed')

# 9. Ablation study

Variasi yang diuji: Full MARL, strategi Planning Balanced tetap, tanpa reward ranking, tanpa reward coverage, dan tanpa bias semantik pada level item. Setiap variasi **dilatih ulang** dengan DKT yang sama. Interpretasi `no_planning` tidak menghapus KC/Item Agent.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    if MODE=='ablation':
        tab=RESULTS[RESULTS.subset=='audit'].groupby('model')[['kc_available','kc_global','auc','bce']].agg(['mean','std'])
        display(tab.round(5));tab.to_csv(TABLES/'ablation_mean_std.csv')
        full=RESULTS[RESULTS.model=='marl'].set_index(['seed','subset'])
        impacts=[]
        for _,r in RESULTS[RESULTS.model!='marl'].iterrows():
            base=full.loc[(r.seed,r.subset)]
            impacts.append({'seed':r.seed,'subset':r.subset,'variant':r.model,
              'delta_kc':r.kc_available-base.kc_available,'delta_auc':r.auc-base.auc,
              'delta_bce':r.bce-base.bce})
        display(pd.DataFrame(impacts).round(5))
        pd.DataFrame(impacts).to_csv(TABLES/'ablation_delta_vs_full.csv',index=False)

# 10. Pembekuan model (*model freeze*)

## 10.1 Pembekuan sebelum final test

Jalankan `MODE='freeze'` **setelah full training selesai**, dengan `FINAL_TEST_UNLOCK=False`. File manifest memuat checksum checkpoint DKT dan PPO full seed 42 serta identitas dataset/split. Mode ini **tidak** melatih atau mengevaluasi test.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    FREEZE_PATH=WORKSPACE/'model_freeze_manifest.json'
    def full_checkpoint_files():
        out={'dkt':DKT_BEST}
        for model in MODELS:
            out[model]=WORKSPACE/'runs'/'full'/f'seed{SEED}'/'ppo'/f'{model}_best.pt'
        return out

    if MODE=='freeze':
        assert not FINAL_TEST_UNLOCK
        if FREEZE_PATH.exists():
            raise FileExistsError('Freeze manifest sudah dibuat. Jangan ditimpa! '+str(FREEZE_PATH))
        files=full_checkpoint_files()
        for label,p in files.items():
            if not p.exists():raise FileNotFoundError('Checkpoint full hilang: '+str(p))
        manifest={'data_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'difficulty_sha256':DIFFICULTY_SHA,
          'seed':SEED,'dkt_signature':DKT_SIGNATURE,'config':asdict(C),
          'files':{k:{'relative_path':str(v.relative_to(WORKSPACE)),'sha256':sha256(v)} for k,v in files.items()},
          'frozen_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),
          'test_previously_exposed_in_prior_study':True,
          'note':'Dataset test historis pernah dievaluasi sebelumnya; tidak boleh disebut sepenuhnya independen.'}
        save_json(FREEZE_PATH,manifest)
        print('FREEZE BERHASIL:',FREEZE_PATH)

# 11. Final test terkunci

## 11.1 Prasyarat pembukaan

Final test hanya dievaluasi jika `MODE='final'` **dan** `FINAL_TEST_UNLOCK=True`. Model DKT dan PPO dimuat dari checkpoint full seed 42; checksum diverifikasi terhadap manifest. Tidak ada training atau pemilihan hyperparameter pada mode ini. Jika dataset/test lama telah dilihat, labelkan hasil sebagai **evaluasi lanjutan historis**, bukan *fresh independent holdout*.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    FINAL_ROWS={}
    if MODE=='final':
        if not FINAL_TEST_UNLOCK:raise RuntimeError('Final test terkunci. Selesaikan freeze dahulu.')
        if not FREEZE_PATH.exists():raise FileNotFoundError('Freeze manifest belum ada!')
        manifest=json.loads(FREEZE_PATH.read_text())
        for a,b in [('data_sha256',DATASET_SHA),('split_sha256',SPLIT_SHA),('difficulty_sha256',DIFFICULTY_SHA),('dkt_signature',DKT_SIGNATURE)]:
            if manifest[a]!=b: raise RuntimeError('Mismatch '+a+' pada freeze manifest')
        for name,rec in manifest['files'].items():
            p=WORKSPACE/rec['relative_path']
            if not p.exists() or sha256(p)!=rec['sha256']:raise RuntimeError(f'Checkpoint berubah: {name}')
        DKT.load_state_dict(torch.load(WORKSPACE/manifest['files']['dkt']['relative_path'],map_location=DEVICE,weights_only=True))
        DKT.eval()
        models={}
        for name in MODELS:
            p=Policy(name).to(DEVICE)
            p.load_state_dict(torch.load(WORKSPACE/manifest['files'][name]['relative_path'],map_location=DEVICE,weights_only=True))
            p.eval();models[name]=p
        for name,policy in models.items():
            FINAL_ROWS[name]=evaluate(policy,get_cases('test'))
        final=pd.DataFrame([{'model':name,**summarize(rows)} for name,rows in FINAL_ROWS.items()])
        FINAL_DIR=WORKSPACE/'final_holdout'
        FINAL_DIR.mkdir(parents=True,exist_ok=True)
        final.to_csv(FINAL_DIR/'final_test_comparison.csv',index=False)
        display(final.round(5))
        # ID dipseudonimkan dengan hash konstan. Simpan prediksi untuk audit statistik (akses terbatas).
        rec=[]
        for name,rows in FINAL_ROWS.items():
            for row in rows:
                rec.append({'model':name,'student_hash':hashlib.sha256(row['sid'].encode()).hexdigest()[:16],
                  'kc_available':row['kc_available'],'kc_global':row['kc_global'],
                  'auc_student':row['auc_student'],'bce_student':row['bce'],'length':row['length'],
                  'target_y_json':json.dumps(row['y'].astype(int).tolist()),
                  'pred_p_json':json.dumps(row['p'].astype(float).tolist())})
        pd.DataFrame(rec).to_csv(FINAL_DIR/'per_student_predictions_restricted.csv',index=False)
        print('Hasil final:',FINAL_DIR)
    else:print('Final test tidak dijalankan di mode',MODE)

## 11.2 Paired bootstrap 95%, Wilcoxon, dan koreksi pengujian berganda

Bootstrap dilakukan pada **mahasiswa yang sama** untuk kedua model, kemudian AUC dan BCE dihitung ulang dari prediksi target gabungan. CI dinyatakan sebagai MARL − baseline. Wilcoxon diterapkan pada metrik per mahasiswa; **AUC mahasiswa tidak identik dengan pooled AUC**. Karena ada enam uji konfirmatori dasar (dua baseline × tiga metrik), p-value harus diinterpretasi dengan koreksi yang ditetapkan sebelum pengujian.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    def metric_from_rows(rows,metric):
        if metric=='kc_available':return float(np.mean([r['kc_available'] for r in rows]))
        y=np.concatenate([r['y'] for r in rows]);p=np.concatenate([r['p'] for r in rows])
        if metric=='bce':return float(np.mean(-y*np.log(p)-(1-y)*np.log(1-p)))
        if metric=='auc':return float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan
        raise KeyError(metric)

    def bootstrap_pair(a,b,metric,nboot=2000,seed=42):
        assert [r['sid'] for r in a]==[r['sid'] for r in b]
        assert all(np.array_equal(x['y'],y['y']) for x,y in zip(a,b))
        rng=np.random.default_rng(seed)
        n=len(a);diffs=[]
        for i in range(nboot):
            ids=rng.integers(0,n,n)
            x=[a[j] for j in ids];y=[b[j] for j in ids]
            va=metric_from_rows(x,metric);vb=metric_from_rows(y,metric)
            if np.isfinite(va) and np.isfinite(vb):diffs.append(va-vb)
        if not diffs:return (np.nan,np.nan)
        return tuple(np.quantile(diffs,[.025,.975]).astype(float))

    if MODE=='final':
        stat=[]
        for base in ('rl_flat','rl_feat'):
            for metric in ('kc_available','auc','bce'):
                a=FINAL_ROWS['marl'];b=FINAL_ROWS[base]
                observed=metric_from_rows(a,metric)-metric_from_rows(b,metric)
                lo,hi=bootstrap_pair(a,b,metric,C.bootstrap_n,SEED)
                if metric=='auc':
                    pairs=[(x['auc_student'],y['auc_student']) for x,y in zip(a,b)
                           if np.isfinite(x['auc_student']) and np.isfinite(y['auc_student'])]
                    paired=np.asarray([x-y for x,y in pairs])
                    unit='student AUC (bukan pooled AUC)'
                elif metric=='kc_available':
                    paired=np.asarray([x['kc_available']-y['kc_available'] for x,y in zip(a,b)])
                    unit='student KC'
                else:
                    paired=np.asarray([x['bce']-y['bce'] for x,y in zip(a,b)])
                    unit='student BCE'
                p=float(wilcoxon(paired,alternative='two-sided').pvalue) if len(paired) and not np.allclose(paired,0) else 1.0
                stat.append({'comparison':'marl vs '+base,'metric':metric,'difference':observed,
                      'bootstrap_ci_low':lo,'bootstrap_ci_high':hi,'wilcoxon_p':p,
                      'wilcoxon_unit':unit,'n_students':len(paired)})
        stats=pd.DataFrame(stat)
        # Holm correction across six exploratory Wilcoxon tests.
        order=np.argsort(stats.wilcoxon_p.to_numpy());n=len(stats)
        adjusted=np.empty(n);running=0.0
        for rank,idx in enumerate(order):
            running=max(running,min(1.0,(n-rank)*stats.loc[idx,'wilcoxon_p']))
            adjusted[idx]=running
        stats['wilcoxon_p_holm']=adjusted
        stats.to_csv(FINAL_DIR/'paired_statistics.csv',index=False)
        display(stats.round(5))
        fig,axs=plt.subplots(1,3,figsize=(13.5,3.5))
        for ax,m in zip(axs,('kc_available','auc','bce')):
            ax.bar(final.model,final[m]);ax.set_title(m+' (↑)' if m!='bce' else m+' (↓)')
            ax.grid(axis='y',alpha=.2)
        fig.tight_layout();fig.savefig(FINAL_DIR/'final_test_comparison.png',dpi=170)
        fig.savefig(FINAL_DIR/'final_test_comparison.svg')
        display(fig);plt.close(fig)

# 12. Ringkasan dan dokumentasi penelitian

## 12.1 Struktur keluaran

```text
MyDrive/Riset_MARL/
├── input/                       # ZIP DBE-KT22 asli
└── outputs/
    ├── dkt/smoke/                # DKT 2 epoch + progress
    ├── dkt/full/                 # DKT 15 epoch + progress
    ├── runs/smoke/               # PPO dan metrik smoke
    ├── runs/full/                # PPO 50 update dan metrik full
    ├── runs/multiseed/           # PPO lima seed dan ringkasannya
    ├── runs/ablation/            # Eksperimen kontribusi komponen
    ├── model_freeze_manifest.json
    └── final_holdout/            # Hanya bila final dibuka
```

## 12.2 Kriteria kesimpulan penelitian

MARL dapat disebut unggul **secara numerik pada ketiga metrik** hanya jika KC Available serta pooled AUC lebih besar daripada kedua baseline, sementara BCE lebih kecil daripada keduanya, pada subset evaluasi yang sama. Keunggulan **secara statistik** menuntut ketidakpastian yang mendukung klaim dan koreksi multipel yang sesuai. Jangan menilai hasil hanya dari smoke atau memilih seed terbaik berdasarkan final test.

### Keterbatasan wajib

1. Evaluasi offline replay menggunakan respons yang teramati; respons untuk soal yang belum pernah dikerjakan tidak tersedia.
2. Hyperparameter dan desain ini dikembangkan setelah hasil penelitian pertama dilihat; dataset test historis tidak sepenuhnya independen.
3. KC Available dan KC Global mengukur besaran berbeda; keduanya dilaporkan.
4. MARL menggunakan kebijakan hierarkis dengan critic bersama, bukan implementasi MAPPO/CTDE penuh tanpa syarat.
5. Training baru dari awal, termasuk difficulty train-only, **tidak menjamin mengulang angka smoke terdahulu**.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    report={
     'mode':MODE,'dataset_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,
     'dkt_checkpoint_sha256':sha256(DKT_BEST),'output_root':str(WORKSPACE),
     'model_types':['rl_flat','rl_feat','marl'],
     'full_training_required_before_final':True,
     'final_test_locked':MODE!='final',
     'known_prior_test_exposure':True}
    if not RESULTS.empty:
        report['n_records']=int(len(RESULTS))
        if not WIN_REPORT.empty:report['win_records']=WIN_REPORT.to_dict(orient='records')
    save_json(REPORTS/'run_summary.json',report)
    print('SELESAI MODE:',MODE,'\nOutput:',MODE_ROOT)

# Audit pembanding dengan informasi KC setara

Mode `kc_smoke` memeriksa alur dengan anggaran pelatihan kecil. Mode `kc_audit` melatih kebijakan pada lima seed dan mengevaluasi kelompok policy audit. Keduanya membaca ZIP dari `MyDrive/Riset_MARL/input/` dan menulis ke `MyDrive/Riset_MARL/outputs_kc_comparison/`.

`rl_kc` memilih soal secara langsung. Pembanding ini memakai skor KC dari strategi Balanced, dirata-ratakan pada KC yang terkait dengan setiap soal, serta bias item dan prior bersama. Ini menyetarakan sumber informasi dan komponen bias. Besar efek prior dapat berbeda karena MARL menormalkan peluang pada tiap tahap dan RL-KC menormalkan peluang langsung pada seluruh soal layak.

Hasil audit lama tetap menjadi acuan historis. Mode baru mengevaluasi kebijakan hasil pelatihan ulang pada split audit yang sama.

In [ ]:
if MODE in ('kc_smoke', 'kc_audit'):
    class GreedyKCCoverage:
        def eval(self):
            return self

        def choice(self, obs, greedy=True):
            eligible = np.flatnonzero(obs['mask_q'])
            gains = qkc[eligible][:, ~obs['covered']].sum(axis=1)
            question = int(eligible[np.argmax(gains)])  # Seri: ID soal terkecil.
            return {'q': question}, None, None

    audit_cases = get_cases('audit')
    if C.smoke:
        audit_cases = audit_cases[:20]
    seeds = [SEED] if C.smoke else MULTI_SEEDS
    greedy_rows = evaluate(GreedyKCCoverage(), audit_cases)
    records = []
    jobs = [('rl_flat', 'none'), ('rl_feat', 'none'), ('rl_kc', 'none'),
            ('marl', 'none'), ('marl', 'no_planning'), ('marl', 'no_special_priors')]
    for seed in seeds:
        records.append({'seed': seed, 'model': 'greedy_kc', **summarize(greedy_rows)})
        for kind, ablation in jobs:
            policy, history = train_policy(kind, seed, ablation)
            rows = evaluate(policy, audit_cases)
            assert [r['sid'] for r in rows] == [r['sid'] for r in greedy_rows]
            assert all(np.array_equal(a['y'], b['y']) for a, b in zip(rows, greedy_rows))
            label = kind if ablation == 'none' else f'{kind}_{ablation}'
            records.append({'seed': seed, 'model': label, **summarize(rows)})
            pd.DataFrame(records).to_csv(TABLES / 'kc_audit_progress.csv', index=False)

    audit = pd.DataFrame(records)
    audit.to_csv(TABLES / 'kc_audit_by_seed.csv', index=False)
    metrics = ['kc_available', 'kc_global', 'auc', 'bce']
    summary = audit.groupby('model')[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(column) for column in summary.columns]
    summary.to_csv(TABLES / 'kc_audit_summary.csv')
    primary = audit.set_index(['seed', 'model'])[metrics].unstack('model')
    differences = pd.DataFrame({
        metric: primary[metric]['marl'] - primary[metric]['rl_kc']
        for metric in metrics
    })
    differences.to_csv(TABLES / 'marl_minus_rl_kc_by_seed.csv')
    display(audit.round(5), summary.round(5), differences.round(5))


# 13. Laporan hasil terintegrasi — tanpa pelatihan ulang

Mode `report` menyusun kembali hasil yang sudah disimpan selama eksperimen. Seluruh evaluasi menggunakan **file yang dihasilkan sebelumnya**, bukan melakukan inferensi atau training ulang. Bagian ini menghasilkan tabel dan grafik untuk laporan penelitian sekaligus menampilkan setiap hasil langsung di Colab.

**Sumber:** `outputs/runs/full/`, `runs/multiseed/`, `runs/ablation/`, `dkt/full/`, `model_freeze_manifest.json`, dan `final_holdout/`. File keluaran baru ditempatkan di `outputs/laporan/`; sumber tidak diubah.

Untuk menjaga reproduksibilitas, identitas dataset dan split diperiksa sebelum hasil antareksperimen digabungkan. Hasil final berasal dari kelompok test historis yang telah dievaluasi pada penelitian sebelumnya sehingga tidak diklaim sebagai holdout yang sepenuhnya baru.

Tambahan Bab IV: Bagian 13.10–13.14 meliputi distribusi interaksi, difficulty train-only, diagnostik kurva training, serta heatmap multi-seed dan ablasi. Grafik distribusi membutuhkan ZIP asli di `input/`; bagian lainnya menggunakan artefak tersimpan. Seluruh gambar baru ditampilkan dalam notebook dan disimpan ke `outputs/laporan/figures/`.


## 13.1 Memuat dan memeriksa artefak penelitian

Metrik dan riwayat PPO dibaca dari tabel yang sudah ada. Apabila suatu berkas wajib tidak ditemukan, proses pelaporan dihentikan dengan pesan yang menyebutkan lokasi berkas tersebut; tidak ada data pengganti yang dibuat.

In [ ]:
from IPython.display import display, Markdown, Image
from sklearn.metrics import roc_curve, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from matplotlib.ticker import MaxNLocator

if MODE != 'report':
    print('Pelaporan khusus hanya dijalankan pada MODE="report".')
else:
    LAPORAN = WORKSPACE / 'laporan'
    LT, LF, LR = (LAPORAN / x for x in ('tables', 'figures', 'reports'))
    for p in (LT, LF, LR): p.mkdir(parents=True, exist_ok=True)
    SOURCES = {
        'full': WORKSPACE/'runs'/'full',
        'multiseed': WORKSPACE/'runs'/'multiseed',
        'ablation': WORKSPACE/'runs'/'ablation',
        'final': WORKSPACE/'final_holdout',
    }
    REQUIRED = {
        'full': 'tables/model_comparison.csv',
        'multiseed': 'tables/model_comparison.csv',
        'ablation': 'tables/model_comparison.csv',
        'final': 'final_test_comparison.csv',
    }
    for k, rel in REQUIRED.items():
        if not (SOURCES[k]/rel).is_file():
            raise FileNotFoundError(f'Berkas wajib belum tersedia: {SOURCES[k]/rel}. Pastikan eksperimen sebelumnya telah selesai dan hasilnya ada di Drive.')
    DATA = {k: pd.read_csv(SOURCES[k]/rel) for k,rel in REQUIRED.items()}
    FINAL_STATS = pd.read_csv(SOURCES['final']/'paired_statistics.csv')
    FINAL_PRED = pd.read_csv(SOURCES['final']/'per_student_predictions_restricted.csv')
    MANIFEST_FILE = WORKSPACE/'model_freeze_manifest.json'
    if not MANIFEST_FILE.is_file():
        raise FileNotFoundError(f'Manifest pembekuan tidak ditemukan: {MANIFEST_FILE}')
    MANIFEST = json.loads(MANIFEST_FILE.read_text(encoding='utf8'))
    IDENTITIES = {k: json.loads((SOURCES[k]/'reports'/'experiment_identity.json').read_text(encoding='utf8'))
                  for k in ('full','multiseed','ablation')}
    id_fields=('dataset_sha256','split_sha256','difficulty_sha256')
    for key in id_fields:
        vals = {IDENTITIES[k][key] for k in IDENTITIES}
        vals.add(MANIFEST[key] if key != 'dataset_sha256' else MANIFEST['data_sha256'])
        if len(vals)!=1: raise RuntimeError(f'Identitas eksperimen tidak cocok: {key}. Jangan menggabungkan hasil.')
    assert set(DATA['final'].model)=={'rl_flat','rl_feat','marl'}
    assert len(FINAL_PRED.groupby(['model','student_hash']))==len(FINAL_PRED)
    checksum_rows=[]
    for name,record in MANIFEST['files'].items():
        p=WORKSPACE/record['relative_path']
        status='COCOK' if p.is_file() and sha256(p)==record['sha256'] else 'FILE_TIDAK_ADA_ATAU_BERUBAH'
        checksum_rows.append({'model':name,'status':status,'path':record['relative_path']})
    CHECKSUM_DF=pd.DataFrame(checksum_rows)
    CHECKSUM_DF.to_csv(LT/'checkpoint_integrity.csv',index=False)
    display(Markdown('**Identitas dataset dan split konsisten pada full training, multi-seed, ablasi, dan freeze manifest.**'))
    display(CHECKSUM_DF)
    if not (CHECKSUM_DF.status=='COCOK').all():
        display(Markdown('**Catatan:** sebagian checkpoint tidak dapat diverifikasi dari jalur Drive ini. Grafik laporan tetap berasal dari CSV tersimpan; pemuatan ulang checkpoint tidak dilakukan.'))
    else:
        display(Markdown('Keempat checkpoint pembekuan terverifikasi tanpa mengubah isinya.'))
    df_counts=IDENTITIES['full']['students']
    profile=pd.DataFrame([{'kelompok':k,'mahasiswa':v} for k,v in df_counts.items()])
    profile.to_csv(LT/'komposisi_data.csv',index=False)
    display(profile)
    print('Sumber hasil: ',*(f'{key}: {p}' for key,p in SOURCES.items()),sep='\n- ')

## 13.2 Dataset, pembagian mahasiswa, dan arsitektur sistem

DBE-KT22 dibagi berdasarkan identitas mahasiswa untuk memisahkan pelatihan DKT, pemilihan checkpoint, audit kebijakan, dan evaluasi akhir. Pemilihan soal dilakukan melalui replay respons yang tercatat, sehingga hasil eksperimen merepresentasikan simulasi CAT offline, bukan percobaan tes adaptif pada mahasiswa secara langsung.

In [ ]:
if MODE == 'report':
    model_labels={'rl_flat':'RL-Flat','rl_feat':'RL-Feat','marl':'MARL'}
    figure_colors={'rl_flat':'#4E79A7','rl_feat':'#F28E2B','marl':'#59A14F'}
    def show_save(fig, basename):
        fig.tight_layout()
        fig.savefig(LF/f'{basename}.png',dpi=190,bbox_inches='tight')
        fig.savefig(LF/f'{basename}.svg',bbox_inches='tight')
        display(fig)
        plt.close(fig)
    counts=IDENTITIES['full']['students']
    order=['train','dkt_val','policy_select','policy_audit','test']
    fig,ax=plt.subplots(figsize=(10,3.4))
    heights=[counts[k] for k in order]
    bars=ax.bar([k.replace('_',' ') for k in order],heights,color=['#4E79A7','#9C755F','#76B7B2','#59A14F','#B07AA1'])
    ax.bar_label(bars,padding=3)
    ax.set_ylabel('Jumlah mahasiswa');ax.set_title('Pembagian data penelitian DBE-KT22')
    ax.grid(axis='y',alpha=.15)
    show_save(fig,'01_split_mahasiswa')
    import matplotlib.patches as patches
    fig,ax=plt.subplots(figsize=(11.5,4.3));ax.set(xlim=(0,12),ylim=(0,4));ax.axis('off')
    boxes=[(0.3,1.35,2,1.2,'Riwayat respons\nDBE-KT22','#DCEAF5'),
           (2.9,1.35,2.2,1.2,'DKT–GRU\nState dan prediksi','#DCEAF5'),
           (5.8,2.55,2.3,0.8,'Planning Agent','#DFF2E3'),
           (5.8,1.4,2.3,0.8,'KC Selection Agent','#DFF2E3'),
           (5.8,0.25,2.3,0.8,'Item Selection Agent','#DFF2E3'),
           (8.7,1.35,2.9,1.2,'CAT Offline\n20 butir, reward, evaluasi','#F3E6D5')]
    for x,y,w,h,t,fc in boxes:
        ax.add_patch(patches.FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.09',fc=fc,ec='#49545A',lw=1.2))
        ax.text(x+w/2,y+h/2,t,ha='center',va='center',fontsize=10)
    props=dict(arrowstyle='->',color='#52616B',lw=1.5)
    for a,b in [((2.35,1.95),(2.83,1.95)),((5.17,1.95),(5.7,2.95)),((6.95,2.53),(6.95,2.22)),((6.95,1.37),(6.95,1.07)),((8.2,0.65),(8.6,1.75))]:
        ax.annotate('',xy=b,xytext=a,arrowprops=props)
    ax.set_title('Arsitektur konseptual DKT–MARL (tiga kebijakan kooperatif)',pad=12,fontsize=13)
    show_save(fig,'02_arsitektur_dkt_marl')
    display(Markdown('**Interpretasi.** DKT menyediakan estimasi kondisi mahasiswa. Planning Agent menetapkan strategi, KC Selection Agent memilih konsep, dan Item Selection Agent memilih soal. Ketiga keputusan dioptimalkan dengan PPO; gambar ini menunjukkan alur konseptual, bukan hasil pengamatan tiap aksi.'))

## 13.3 Kurva pelatihan DKT

Kinerja DKT dideskripsikan berdasarkan *training BCE* dan *validation BCE*. Jika riwayat epoch tersedia, kedua kurva digambar ulang. Jika tidak, kurva asli dari hasil full training tetap ditampilkan. Perubahan performa selama training tidak dihitung dari checkpoint yang telah dibekukan.

In [ ]:
if MODE == 'report':
    hfile=WORKSPACE/'dkt'/'full'/'dkt_history.csv'
    if hfile.is_file():
        hist=pd.read_csv(hfile)
        assert {'epoch','train_batch_bce','val_bce'}.issubset(hist.columns)
        display(hist.round(5))
        hist.to_csv(LT/'dkt_riwayat.csv',index=False)
        fig,ax=plt.subplots(figsize=(10,4))
        ax.plot(hist.epoch,hist.train_batch_bce,marker='o',label='Training BCE')
        ax.plot(hist.epoch,hist.val_bce,marker='o',label='Validation BCE')
        i=int(hist.val_bce.idxmin());best_ep=int(hist.iloc[i].epoch)
        ax.axvline(best_ep,ls='--',alpha=.5)
        ax.set(xlabel='Epoch',ylabel='BCE',title=f'Pembelajaran DKT — checkpoint terbaik epoch {best_ep}')
        ax.legend();ax.grid(alpha=.2);show_save(fig,'03_kurva_dkt')
        display(Markdown(f'**Interpretasi.** Validation BCE terendah tercatat pada epoch ke-{best_ep} sebesar {hist.iloc[i].val_bce:.4f}. Checkpoint terbaik dipilih dari validation, bukan dari kelompok final test.'))
    else:
        src=SOURCES['full']/'figures'/'01_dkt_learning_curve.png'
        if src.is_file():
            import shutil
            shutil.copy2(src,LF/'03_kurva_dkt.png')
            svg=SOURCES['full']/'figures'/'01_dkt_learning_curve.svg'
            if svg.is_file():shutil.copy2(svg,LF/'03_kurva_dkt.svg')
            display(Image(filename=str(src)))
            display(Markdown('**Catatan.** Grafik DKT ditampilkan dari keluaran training sebelumnya; CSV epoch lengkap belum ditemukan pada `dkt/full/`. Angka epoch tidak direkonstruksi dari gambar.'))
        else:display(Markdown('**Data DKT:** riwayat dan grafik tidak tersedia di Drive.'))

## 13.4 Pelatihan PPO dan pemilihan checkpoint

Riwayat lima puluh update digunakan untuk melihat perubahan utility, KC Coverage, AUC, dan BCE sepanjang pelatihan. Nilai terbaik berasal dari mekanisme pemilihan checkpoint yang sudah ditetapkan pada eksperimen; visualisasi tidak memilih model baru.

In [ ]:
if MODE == 'report':
    histories={}
    best_rows=[]
    for model in ('rl_flat','rl_feat','marl'):
        p=SOURCES['full']/f'seed42/{model}_history.csv'
        if not p.is_file():raise FileNotFoundError(p)
        h=pd.read_csv(p);histories[model]=h
        v=h.dropna(subset=['utility_val'])
        if v.empty:raise ValueError(f'Utility validasi kosong: {p}')
        best=v.loc[v.utility_val.idxmax()]
        best_rows.append({'model':model,'update_terpilih':int(best['update']),'utility':best.utility_val,
                          'kc_available':best.kc_available,'auc':best.auc,'bce':best.bce,'total_update':len(h)})
    best_df=pd.DataFrame(best_rows);display(best_df.round(5));best_df.to_csv(LT/'ppo_checkpoint_terpilih.csv',index=False)
    fig,axs=plt.subplots(2,2,figsize=(12,7))
    for ax,(metric,name) in zip(axs.ravel(),[('utility_val','Validation utility'),('kc_available','KC Available ↑'),('auc','AUC ↑'),('bce','BCE ↓')]):
        for model,h in histories.items():
            v=h.dropna(subset=[metric]);ax.plot(v['update'],v[metric],label=model_labels[model],lw=1.7,color=figure_colors[model])
        ax.set(xlabel='PPO update',ylabel=name,title=name);ax.grid(alpha=.18)
    axs[0,0].legend();show_save(fig,'04_kurva_ppo_full_seed42')
    m=best_df.set_index('model').loc['marl'];display(Markdown(f'**Interpretasi.** Checkpoint MARL yang digunakan pada eksperimen seed 42 berasal dari update ke-{int(m.update_terpilih)} dari {int(m.total_update)} update. Performa setelah titik ini tidak selalu lebih baik; hasil full training dihitung dari checkpoint terpilih, bukan otomatis dari update terakhir.'))

## 13.5 Evaluasi *policy selection* dan *policy audit*

Perbandingan pada kelompok pemilihan model dan audit disajikan terpisah. Pada setiap metrik, hasil terbaik ditentukan menggunakan arah optimasi yang benar: KC/AUC lebih besar lebih baik, sedangkan BCE lebih kecil lebih baik.

In [ ]:
if MODE == 'report':
    full=DATA['full'].copy()
    display(full.round(5));full.to_csv(LT/'full_selection_audit.csv',index=False)
    fig,axes=plt.subplots(2,3,figsize=(13,7))
    for row,subset in enumerate(('selection','audit')):
        part=full[full.subset==subset].set_index('model').reindex(['rl_flat','rl_feat','marl'])
        for col,(metric,title) in enumerate([('kc_available','KC Available ↑'),('auc','AUC ↑'),('bce','BCE ↓')]):
            ax=axes[row,col];bars=ax.bar([model_labels[x] for x in part.index],part[metric],
                             color=[figure_colors[x] for x in part.index]);
            ax.bar_label(bars,fmt='%.4f',fontsize=8,padding=2)
            lower=max(0,part[metric].min()-0.035);upper=part[metric].max()+0.035
            ax.set_ylim(lower,upper);ax.set_title(f'{subset.capitalize()} — {title}');ax.grid(axis='y',alpha=.15)
    show_save(fig,'05_perbandingan_full_selection_audit')
    for subset in ('selection','audit'):
        t=full[full.subset==subset].set_index('model')
        win={m: ('MARL' if (t.loc['marl',m]>(t.loc[['rl_flat','rl_feat'],m].max()) if m!='bce' else
                                    t.loc['marl',m]<(t.loc[['rl_flat','rl_feat'],m].min())) else 'baseline')
               for m in ('kc_available','auc','bce')}
        display(Markdown(f'**{subset.capitalize()}:** KC = {win["kc_available"]}; AUC = {win["auc"]}; BCE = {win["bce"]}. Perbedaan numerik belum merupakan bukti signifikansi statistik.'))

## 13.6 Evaluasi lima seed

Variasi pelatihan PPO dianalisis menggunakan seed 42, 123, 321, 2026, dan 2027. Rata-rata dan standar deviasi dihitung dari masing-masing seed, bukan dari prediksi mahasiswa yang digabung. Jumlah kemenangan seluruh metrik menunjukkan konsistensi per seed dan tidak setara dengan uji signifikansi.

In [ ]:
if MODE == 'report':
    multi=DATA['multiseed'].copy()
    assert set(multi.seed)=={42,123,321,2026,2027} and len(multi)==30
    means=multi.groupby(['subset','model'])[['kc_available','kc_global','auc','bce']].agg(['mean','std'])
    means_tidy = means.copy()
    means_tidy.columns = [f'{metric}_{stat}' for metric, stat in means_tidy.columns]
    means_tidy.reset_index().to_csv(LT/'multiseed_mean_std.csv', index=False)
    display(means.round(5))
    fig,axs=plt.subplots(2,3,figsize=(13,7),sharex=False)
    for ri,subset in enumerate(('selection','audit')):
        part=multi[multi.subset==subset]
        for col,metric in enumerate(('kc_available','auc','bce')):
            ax=axs[ri,col]
            for model in ('rl_flat','rl_feat','marl'):
                s=part[part.model==model].sort_values('seed')
                ax.plot([str(x) for x in s.seed],s[metric],marker='o',lw=1.6,color=figure_colors[model],label=model_labels[model])
            ax.set_title(f'{subset} — {metric}');ax.grid(alpha=.2)
    axs[0,0].legend(fontsize=8);show_save(fig,'06_multiseed_per_seed')
    win=[]
    for subset,frame in multi.groupby('subset'):
        for seed,s in frame.groupby('seed'):
            t=s.set_index('model')
            row={'subset':subset,'seed':int(seed)}
            for metric in ('kc_available','kc_global','auc','bce'):
                best=t.loc[['rl_flat','rl_feat'],metric]
                row[metric+'_win']=bool(t.loc['marl',metric]>best.max() if metric!='bce' else t.loc['marl',metric]<best.min())
            row['win_kc_auc_bce']=row['kc_available_win'] and row['auc_win'] and row['bce_win']
            win.append(row)
    wins=pd.DataFrame(win).sort_values(['subset','seed']);wins.to_csv(LT/'kemenangan_per_seed.csv',index=False)
    display(wins)
    win_agg=wins.groupby('subset')[['kc_available_win','auc_win','bce_win','win_kc_auc_bce']].sum()
    display(win_agg)
    for subset in ('audit','selection'):
        s=win_agg.loc[subset]
        display(Markdown(f'**{subset}:** MARL unggul KC pada {int(s.kc_available_win)}/5 seed, AUC pada {int(s.auc_win)}/5, BCE pada {int(s.bce_win)}/5, dan ketiganya sekaligus pada {int(s.win_kc_auc_bce)}/5 seed.'))

## 13.7 Studi ablasi tiga agen

Lima konfigurasi dibandingkan pada dua seed. Varian `No Planning` menggunakan strategi Balanced tetap; pengujian ini **tidak** menghapus KC Selection Agent maupun Item Selection Agent. Perbandingan menggambarkan kontribusi komponen secara eksploratif, bukan bukti kausal yang sudah signifikan.

In [ ]:
if MODE == 'report':
    abl=DATA['ablation'];audit=abl[abl.subset=='audit']
    variant_map={'marl':'Full MARL','marl_no_planning':'No Planning','marl_no_rank_reward':'No Rank Reward',
                 'marl_no_coverage_reward':'No Coverage Reward','marl_no_semantic_prior':'No Semantic Prior'}
    summary=audit.groupby('model')[['kc_available','kc_global','auc','bce']].agg(['mean','std'])
    summary_tidy=summary.copy()
    summary_tidy.columns=[f'{metric}_{stat}' for metric,stat in summary_tidy.columns]
    summary_tidy.reset_index().to_csv(LT/'ablation_mean_std_audit.csv',index=False)
    display(summary.round(5))
    diffs=[]
    for seed,t in audit.groupby('seed'):
        ref=t[t.model=='marl'].iloc[0]
        for _,r in t.iterrows():
            if r.model=='marl':continue
            diffs.append({'seed':int(seed),'variant':r.model,
                          'delta_kc':r.kc_available-ref.kc_available,'delta_auc':r.auc-ref.auc,'delta_bce':r.bce-ref.bce})
    delta=pd.DataFrame(diffs);delta.to_csv(LT/'ablation_delta_audit.csv',index=False);display(delta.round(5))
    df=summary.xs('mean',level=1,axis=1).reindex(variant_map)
    fig,axs=plt.subplots(1,3,figsize=(15,5.0))
    for ax,(metric,title) in zip(axs,[('kc_available','KC Available ↑'),('auc','AUC ↑'),('bce','BCE ↓')]):
        ax.barh([variant_map[k] for k in df.index],df[metric],color=['#59A14F']+['#A0AEC0']*4)
        lo,hi=df[metric].min(),df[metric].max();ax.set_xlim(max(0,lo-(hi-lo)*.8),hi+(hi-lo)*.6)
        ax.set_title(title);ax.grid(axis='x',alpha=.2);ax.invert_yaxis()
    show_save(fig,'07_ablation_audit')
    display(Markdown('**Interpretasi.** Penghilangan komponen tidak selalu memperburuk ketiga metrik sekaligus. Ablasi dua seed digunakan untuk menunjukkan perubahan arah dan besarnya metrik, sedangkan klaim keunggulan komponen harus mempertimbangkan variasi eksperimen.'))

## 13.8 Hasil final test dan ketidakpastian statistik

Hasil akhir diambil dari checkpoint yang dibekukan pada seed 42. Perbedaan ditulis sebagai **MARL − baseline**, sehingga arah yang diinginkan adalah positif untuk KC/AUC dan negatif untuk BCE. Interval bootstrap dan nilai p Wilcoxon adalah hasil analisis sebelumnya yang dibaca ulang; tidak dihitung ulang atau disesuaikan untuk menghasilkan kesimpulan tertentu.

In [ ]:
if MODE == 'report':
    fin=DATA['final'].set_index('model').reindex(['rl_flat','rl_feat','marl'])
    display(fin.round(5))
    DATA['final'].to_csv(LT/'final_test_comparison.csv',index=False)
    display(FINAL_STATS.round(6));FINAL_STATS.to_csv(LT/'paired_statistics.csv',index=False)
    fig,axarr=plt.subplots(1,4,figsize=(15,3.7))
    for ax,(metric,title) in zip(axarr,[('kc_available','KC Available ↑'),('kc_global','KC Global ↑'),('auc','AUC ↑'),('bce','BCE ↓')]):
        bars=ax.bar([model_labels[x] for x in fin.index],fin[metric],color=[figure_colors[x] for x in fin.index]);ax.bar_label(bars,fmt='%.4f',fontsize=8,padding=2)
        lo=fin[metric].min();hi=fin[metric].max();ax.set_ylim(max(0,lo-(hi-lo)*.7-0.002),hi+(hi-lo)*.55+0.002)
        ax.set_title(title);ax.grid(axis='y',alpha=.17)
    show_save(fig,'08_final_perbandingan')
    # Interval plot, a zero crossing indicates uncertainty about the sign.
    df=FINAL_STATS.copy()
    df['nama']=df.comparison.str.replace('marl vs ','MARL − ',regex=False)+' / '+df.metric
    fig,ax=plt.subplots(figsize=(11,5.2))
    ys=np.arange(len(df))
    for i,(_,row) in enumerate(df.iterrows()):
        col='#59A14F' if row.metric=='kc_available' else '#4E79A7'
        ax.plot([row.bootstrap_ci_low,row.bootstrap_ci_high],[i,i],color=col,lw=3)
        ax.plot(row.difference,i,marker='o',ms=7,color=col)
    ax.axvline(0,ls='--',color='black',alpha=.65)
    ax.set_yticks(ys,df.nama);ax.invert_yaxis()
    ax.set(xlabel='Selisih MARL − baseline',title='Paired bootstrap CI 95% — hasil final test')
    ax.grid(axis='x',alpha=.2)
    show_save(fig,'09_bootstrap_ci_final')
    kc=FINAL_STATS[FINAL_STATS.metric=='kc_available']
    others=FINAL_STATS[FINAL_STATS.metric!='kc_available']
    display(Markdown('**Interpretasi.** Interval KC Available terhadap kedua baseline berada di atas nol; sedangkan interval AUC dan BCE masih melintasi nol. Karena itu, keunggulan KC lebih kuat secara statistik dibanding keunggulan prediksi. Koreksi Holm pada nilai p Wilcoxon sudah tersedia dalam tabel statistik asli.'))

## 13.9 Pemeriksaan prediksi individual, ROC, dan kalibrasi

Grafik ROC dan kalibrasi dibuat dari `per_student_predictions_restricted.csv`, yaitu probabilitas dan target yang telah tersimpan pada final test. Ini merupakan **visualisasi tambahan atas prediksi lama**, bukan pengujian ulang model. Untuk melindungi data, nama maupun identitas mahasiswa tidak dimasukkan ke grafik.

Kalibrasi memperlihatkan hubungan antara probabilitas prediksi dan proporsi jawaban benar pada data yang sama. Grafik ini bersifat deskriptif; tidak digunakan untuk mengoptimalkan model atau menentukan checkpoint baru.

In [ ]:
if MODE == 'report':
    pred=FINAL_PRED.copy()
    for m in ('rl_flat','rl_feat','marl'):
        d=pred[pred.model==m]
        if len(d)!=157:raise RuntimeError(f'Jumlah mahasiswa final tidak cocok: {m}={len(d)}')
    student_sets=[set(pred[pred.model==m].student_hash) for m in ('rl_flat','rl_feat','marl')]
    if not all(s==student_sets[0] for s in student_sets[1:]):raise RuntimeError('Mahasiswa final berbeda antar model')
    y_by_model={};p_by_model={}
    for model in ('rl_flat','rl_feat','marl'):
        d=pred[pred.model==model].sort_values('student_hash')
        ys=[np.array(json.loads(x),dtype=int) for x in d.target_y_json]
        ps=[np.array(json.loads(x),dtype=float) for x in d.pred_p_json]
        if not all(len(y)==len(p) for y,p in zip(ys,ps)):
            raise RuntimeError('Target dan prediksi tidak sejajar')
        y_by_model[model]=np.concatenate(ys);p_by_model[model]=np.concatenate(ps)
    if not all(np.array_equal(y_by_model['rl_flat'],y_by_model[m]) for m in ('rl_feat','marl')):
        raise RuntimeError('Target diagnosis final tidak sama antar model')
    check=[]
    for m in ('rl_flat','rl_feat','marl'):
        au=roc_auc_score(y_by_model[m],p_by_model[m])
        y=y_by_model[m];p=p_by_model[m]
        p=np.clip(p,1e-7,1-1e-7)
        bce=float(np.mean(-y*np.log(p)-(1-y)*np.log(1-p)))
        check.append({'model':m,'n_target':len(y),'auc_dihitung_ulang':au,'bce_dihitung_ulang':bce,
                      'auc_dilaporkan':float(fin.loc[m,'auc']),'bce_dilaporkan':float(fin.loc[m,'bce'])})
        if abs(au-float(fin.loc[m,'auc']))>1e-5 or abs(bce-float(fin.loc[m,'bce']))>1e-5:
            raise RuntimeError(f'Perhitungan prediksi tidak cocok dengan tabel final: {m}')
    check=pd.DataFrame(check);display(check.round(5));check.to_csv(LT/'rekalkulasi_final.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(11.5,4.5))
    for m in ('rl_flat','rl_feat','marl'):
        fpr,tpr,_=roc_curve(y_by_model[m],p_by_model[m]);axs[0].plot(fpr,tpr,lw=1.8,color=figure_colors[m],label=f'{model_labels[m]} (AUC {roc_auc_score(y_by_model[m],p_by_model[m]):.3f})')
        frac,mean=calibration_curve(y_by_model[m],p_by_model[m],n_bins=10,strategy='quantile')
        axs[1].plot(mean,frac,marker='o',color=figure_colors[m],label=model_labels[m])
    axs[0].plot([0,1],[0,1],'--',color='gray',alpha=.6);axs[0].set(xlabel='False Positive Rate',ylabel='True Positive Rate',title='ROC pada prediksi final')
    axs[1].plot([0,1],[0,1],'--',color='gray',alpha=.6);axs[1].set(xlabel='Probabilitas rata-rata',ylabel='Frekuensi jawaban benar',title='Diagram kalibrasi (10 bin kuantil)')
    for ax in axs:ax.legend(fontsize=8);ax.grid(alpha=.15)
    show_save(fig,'10_roc_dan_kalibrasi')
    # Distribution of paired student-level differences; raw IDs never exported.
    fig,axs=plt.subplots(1,3,figsize=(14,3.7))
    for ax,(metric,label) in zip(axs,[('kc_available','KC Available'),('auc_student','AUC individual'),('bce_student','BCE individual')]):
        a=pred[pred.model=='marl'].set_index('student_hash')[metric]
        for m in ('rl_flat','rl_feat'):
            b=pred[pred.model==m].set_index('student_hash')[metric]
            delta=(a-b).dropna();ax.hist(delta,bins=22,histtype='step',lw=2,color=figure_colors[m],label=f'MARL − {model_labels[m]}')
        ax.axvline(0,ls='--',color='black',alpha=.5);ax.set(xlabel='Selisih per mahasiswa',ylabel='Frekuensi',title=label);ax.grid(axis='y',alpha=.2)
    axs[0].legend(fontsize=8);show_save(fig,'11_distribusi_selisih_mahasiswa')
    display(Markdown('**Interpretasi.** ROC merupakan ringkasan diskriminasi prediksi pada target yang sama; diagram kalibrasi memperlihatkan hubungan probabilitas dan kejadian aktual. Distribusi per mahasiswa menjelaskan variasi pasangan hasil yang tidak terlihat pada rata-rata agregat.'))

## 13.10 Distribusi interaksi DBE-KT22

Analisis frekuensi dihitung dari arsip ZIP DBE-KT22 yang tersedia di `MyDrive/Riset_MARL/input/`. Pembacaan ini **hanya untuk statistik deskriptif**, tanpa memperbarui model, checkpoint, dataset, atau split. Prosedur pembersihan mengikuti langkah penelitian: urutan waktu dan ID transaksi, satu respons pertama per pasangan mahasiswa–soal, pembatasan soal yang memiliki relasi KC, serta mahasiswa dengan sedikitnya 28 interaksi.

Grafik yang dihasilkan mencakup frekuensi interaksi per mahasiswa, frekuensi kemunculan soal, jumlah hubungan soal–KC, dan distribusi jawaban. Statistik individu tidak diekspor. Jika ZIP asli tidak tersedia, bagian ini dilewati; gambar tidak diganti menggunakan data sintetis. Grafik berbasis ZIP hanya disimpan di Drive pribadi dan **tidak otomatis disalin ke repositori publik**.

In [ ]:
if MODE == 'report':
    from zipfile import ZipFile
    from io import BytesIO
    from IPython.display import display, Markdown

    def load_frequency_tables_from_original_zip(input_path, expected_digest):
        sources = sorted(Path(input_path).glob('*.zip'))
        if not sources:
            return None
        if len(sources) != 1:
            raise ValueError('Folder input harus memiliki tepat satu ZIP DBE-KT22 asli.')
        archive = sources[0]
        digest = sha256(archive)
        if digest != expected_digest:
            raise RuntimeError('SHA-256 dataset tidak cocok dengan freeze manifest. Histogram dibatalkan.')
        with ZipFile(archive) as outer:
            nested = [n for n in outer.namelist() if n.endswith('datafiles_100102_csv.zip')]
            if len(nested) != 1:
                raise ValueError('ZIP DBE-KT22 tidak memiliki satu arsip datafiles_100102_csv.zip.')
            with ZipFile(BytesIO(outer.read(nested[0]))) as datazip:
                required_files = {'Transaction.csv','Question_KC_Relationships.csv'}
                if not required_files.issubset(datazip.namelist()):
                    raise ValueError('File transaksi/relasi KC tidak lengkap di ZIP asli.')
                tx = pd.read_csv(BytesIO(datazip.read('Transaction.csv')),
                                 usecols=['id','student_id','question_id','start_time','answer_state'],
                                 low_memory=False)
                rel = pd.read_csv(BytesIO(datazip.read('Question_KC_Relationships.csv')),
                                  usecols=['question_id','knowledgecomponent_id'])
        n_original = len(tx)
        tx['start_time'] = pd.to_datetime(tx['start_time'], utc=True, errors='coerce')
        tx['question_id'] = pd.to_numeric(tx['question_id'], errors='coerce')
        tx['answer_state'] = pd.to_numeric(tx['answer_state'], errors='coerce')
        tx = tx.dropna(subset=['student_id','question_id','start_time','answer_state','id']).copy()
        tx = tx[tx['answer_state'].isin([0,1])]
        tx['question_id'] = tx['question_id'].astype('int64')
        tx = tx.sort_values(['student_id','start_time','id'],kind='stable')
        tx = tx.drop_duplicates(['student_id','question_id'],keep='first')
        rel['question_id'] = pd.to_numeric(rel['question_id'],errors='coerce')
        rel['knowledgecomponent_id'] = pd.to_numeric(rel['knowledgecomponent_id'],errors='coerce')
        rel = rel.dropna(subset=['question_id','knowledgecomponent_id']).copy()
        rel['question_id'] = rel['question_id'].astype('int64')
        rel['knowledgecomponent_id'] = rel['knowledgecomponent_id'].astype('int64')
        qset = set(rel['question_id'].unique()).intersection(set(tx['question_id'].unique()))
        tx = tx[tx['question_id'].isin(qset)].copy()
        lengths = tx.groupby('student_id').size()
        eligible = set(lengths[lengths >= C.test_length+8].index)
        tx = tx[tx['student_id'].isin(eligible)].copy()
        rel = rel[rel['question_id'].isin(qset)].drop_duplicates(['question_id','knowledgecomponent_id'])
        per_student = tx.groupby('student_id').size()
        per_question = tx.groupby('question_id').size().reindex(sorted(qset),fill_value=0)
        q_per_kc = rel.groupby('knowledgecomponent_id')['question_id'].nunique().sort_index()
        interaction_per_kc = rel.merge(per_question.rename('n').reset_index(),on='question_id')
        interaction_per_kc = interaction_per_kc.groupby('knowledgecomponent_id')['n'].sum().reindex(q_per_kc.index,fill_value=0)
        per_q_kcs = rel.groupby('question_id')['knowledgecomponent_id'].nunique()
        return {'transaction_total':n_original,'transactions':tx,'per_student':per_student,
                'per_question':per_question,'per_kc':interaction_per_kc,
                'questions_per_kc':q_per_kc,'kcs_per_question':per_q_kcs,
                'source_sha256':digest}

    def validated_answer_counts(transactions):
        if 'answer_state' not in transactions.columns:
            raise ValueError('Kolom answer_state tidak ditemukan di transaksi.')
        raw = pd.to_numeric(transactions['answer_state'], errors='coerce')
        if raw.isna().any() or not raw.isin([0,1]).all():
            vals = transactions['answer_state'].value_counts(dropna=False).head(8).to_dict()
            raise ValueError(f'Label jawaban bukan biner 0/1; periksa pengkodean: {vals}')
        arr = raw.to_numpy(dtype=np.int64)
        n = np.bincount(arr, minlength=2)[:2]
        if len(transactions) == 0 or int(n.sum()) != len(transactions):
            raise RuntimeError('Distribusi label tidak cocok dengan jumlah interaksi; laporan dibatalkan.')
        return pd.Series(n, index=[0,1], name='frekuensi')

    freq = load_frequency_tables_from_original_zip(INPUT, MANIFEST['data_sha256'])
    if freq is None:
        display(Markdown('**Histogram interaksi tidak dibuat:** ZIP DBE-KT22 tidak ada di `input/`. Unggah ZIP asli yang digunakan dalam penelitian lalu jalankan ulang `MODE="report"`. Tidak diperlukan training ulang.'))
    else:
        def clean_distribution_table(values, variable, bins=24):
            # Ekspor hanya histogram agregat; tidak ada ID mahasiswa atau soal.
            values=np.asarray(values,dtype=float)
            values=values[np.isfinite(values)]
            n_bins=min(int(bins),max(1,len(np.unique(values))))
            counts,edges=np.histogram(values,bins=n_bins)
            return pd.DataFrame({'indikator':variable,'batas_bawah':edges[:-1],
                                 'batas_atas':edges[1:],'frekuensi':counts})

        fig,ax=plt.subplots(figsize=(10,4.6))
        student_len=freq['per_student'].to_numpy()
        ax.hist(student_len,bins=min(35,len(np.unique(student_len))),edgecolor='white',alpha=.88)
        ax.axvline(float(np.median(student_len)),ls='--',color='black',label=f'Median {np.median(student_len):.0f}')
        ax.set(title='Distribusi frekuensi interaksi per mahasiswa',xlabel='Jumlah interaksi unik per mahasiswa',ylabel='Frekuensi mahasiswa')
        ax.grid(axis='y',alpha=.18);ax.legend()
        show_save(fig,'12_distribusi_frekuensi_interaksi')
        display(pd.DataFrame([{'mahasiswa_layak':len(student_len),
                               'interaksi_unik':int(len(freq['transactions'])),
                               'rata_rata':float(np.mean(student_len)),
                               'median':float(np.median(student_len)),
                               'minimum':int(np.min(student_len)),
                               'maksimum':int(np.max(student_len))}]).round(2))
        clean_distribution_table(student_len,'interaksi_mahasiswa',35).to_csv(
            LT/'histogram_frekuensi_interaksi.csv',index=False)

        fig,axs=plt.subplots(1,3,figsize=(15,4.3),constrained_layout=True)
        specs=[('per_question','Jumlah respons per soal','Jumlah soal'),
               ('per_kc','Interaksi terkait KC','Jumlah KC'),
               ('questions_per_kc','Jumlah soal per KC','Jumlah KC')]
        for ax,(field,xlabel,ylabel) in zip(axs,specs):
            v=freq[field].to_numpy(dtype=float)
            ax.hist(v,bins=min(26,len(np.unique(v))),edgecolor='white',alpha=.9)
            ax.set(title=field.replace('_',' ').title(),xlabel=xlabel,ylabel=ylabel)
            ax.grid(axis='y',alpha=.2)
            clean_distribution_table(v,field,26).to_csv(LT/f'histogram_{field}.csv',index=False)
        show_save(fig,'13_frekuensi_soal_dan_kc')

        fig,axs=plt.subplots(1,2,figsize=(11.8,4.0))
        # Audit wajib: jumlah dua label harus identik dengan jumlah transaksi yang lolos filter.
        # np.bincount mencegah label yang berbeda tipe terabaikan diam-diam oleh reindex.
        counts=validated_answer_counts(freq['transactions'])
        bars=axs[0].bar(['Salah (0)','Benar (1)'],counts.values)
        axs[0].bar_label(bars);axs[0].set(ylabel='Jumlah interaksi',title='Distribusi label jawaban pada mahasiswa layak')
        axs[0].grid(axis='y',alpha=.16)
        vals=freq['kcs_per_question'].to_numpy()
        axs[1].hist(vals,bins=np.arange(.5,int(max(vals))+1.6,1),rwidth=.8)
        axs[1].set(xlabel='Jumlah KC yang terkait per soal',ylabel='Frekuensi soal',title='Jumlah KC per butir soal')
        axs[1].xaxis.set_major_locator(MaxNLocator(integer=True));axs[1].grid(axis='y',alpha=.16)
        show_save(fig,'14_label_dan_relasi_soal_kc')
        label_table = pd.DataFrame({'label':['salah','benar'],'jumlah':counts.values,
                                    'proporsi':counts.values / counts.sum()})
        if int(label_table['jumlah'].sum()) != len(freq['transactions']):
            raise RuntimeError('Validasi distribusi label gagal: jumlah respons != jumlah interaksi.')
        label_table.to_csv(LT/'distribusi_label_jawaban.csv',index=False)
        display(label_table)
        clean_distribution_table(vals,'kc_per_soal',12).to_csv(LT/'histogram_kc_per_soal.csv',index=False)
        frequency_summary={'n_transaksi_mentah':freq['transaction_total'],
                           'n_interaksi_setelah_filter':int(len(freq['transactions'])),
                           'n_mahasiswa_layak':int(len(student_len)),
                           'n_soal_relatif_tercakup':int(len(freq['per_question'])),
                           'n_kc':int(len(freq['per_kc'])),
                           'sumber_sha256':freq['source_sha256'],
                           'privasi':'Tidak menyimpan student_id, question_id, KC ID, atau respons individu.'}
        save_json(LR/'ringkasan_frekuensi_dataset.json',frequency_summary)
        display(Markdown('**Interpretasi.** Histogram memperlihatkan keragaman panjang riwayat mahasiswa dan ketidakseimbangan frekuensi antarsoal/KC. Interaksi yang terkait beberapa KC dihitung pada masing-masing KC, sehingga jumlahnya tidak harus sama dengan jumlah respons unik. Diagram ini bersifat deskriptif, bukan bukti keberhasilan agen MARL.'))

## 13.11 Difficulty *train-only* dan komposisi split

Kesulitan soal dihitung menggunakan **respons pada prefix mahasiswa training**, bukan respons kelompok selection, audit, atau final test. Perhitungan ini hanya dijalankan jika arsip transaksi dan `split_manifest_private.json` asli tersedia. Hasil yang disimpan adalah ringkasan dan histogram agregat; identitas mahasiswa tidak ikut diekspor.

In [ ]:
if MODE == 'report':
    if 'freq' not in globals() or freq is None:
        print('Distribusi difficulty dilewati: dataset asli tidak tersedia di input/.')
    else:
        split_file=WORKSPACE/'runs'/'full'/'tables'/'split_manifest_private.json'
        if not split_file.is_file():
            display(Markdown('**Distribusi difficulty train-only dilewati:** split manifest privat tidak ditemukan; tidak memakai seluruh respons untuk menggantikannya.'))
        else:
            sp=json.loads(split_file.read_text(encoding='utf-8'))
            train_id={str(x) for x in sp['student_ids']['train']}
            tx=freq['transactions'].copy()
            tx['_sid'] = tx['student_id'].astype(str)
            tx['_n'] = tx.groupby('student_id')['question_id'].transform('size')
            tx['_pos'] = tx.groupby('student_id').cumcount()
            tx['_targets'] = tx['_n'].map(lambda n:max(8,int(round(C.target_frac*int(n)))))
            active=tx[tx['_sid'].isin(train_id)&(tx['_pos'] < tx['_n'] - tx['_targets'])]
            if active.empty:
                raise RuntimeError('Prefix training kosong: periksa kecocokan split dan transaksi.')
            n=active.groupby('question_id').size()
            correct=active.groupby('question_id')['answer_state'].sum()
            questions=freq['per_question'].index
            n=n.reindex(questions,fill_value=0).to_numpy(dtype=float)
            correct=correct.reindex(questions,fill_value=0).to_numpy(dtype=float)
            # Formula sama dengan protokol penelitian (Laplace smoothing).
            difficulty=((n-correct)+2.0)/(n+4.0)
            assert np.isfinite(difficulty).all() and np.all((0<=difficulty)&(difficulty<=1))
            fig,axs=plt.subplots(1,2,figsize=(11.5,4.3))
            axs[0].hist(difficulty,bins=20,edgecolor='white')
            axs[0].set(title='Difficulty berbasis training prefix',xlabel='Estimasi difficulty (0–1)',ylabel='Jumlah soal')
            axs[0].grid(axis='y',alpha=.18)
            axs[1].hist(n,bins=25,edgecolor='white')
            axs[1].set(title='Jumlah respons training per soal',xlabel='Respons training prefix per soal',ylabel='Jumlah soal')
            axs[1].grid(axis='y',alpha=.18)
            show_save(fig,'15_difficulty_train_only')
            clean_distribution_table(difficulty,'difficulty_train_only',20).to_csv(LT/'histogram_difficulty_train_only.csv',index=False)
            clean_distribution_table(n,'n_respons_training_per_soal',25).to_csv(LT/'histogram_respons_train_per_soal.csv',index=False)
            display(pd.DataFrame([{'n_soal':len(difficulty), 'soal_tanpa_respons_train':int((n==0).sum()),
                                   'difficulty_rata_rata':round(float(np.mean(difficulty)),4),
                                   'difficulty_median':round(float(np.median(difficulty)),4)}]))
            display(Markdown('**Interpretasi.** Distribusi difficulty diperoleh hanya dari data training dengan smoothing. Nilai 0,5 pada soal tanpa respons training merupakan nilai prior, bukan estimasi kesulitan yang terukur. Grafik ini tidak mengubah fitur kesulitan pada checkpoint yang telah dibekukan.'))

## 13.12 Diagnostik dinamika training DKT dan PPO

Kurva utama DKT dan PPO sudah tersedia pada Bagian 13.3–13.4. Bagian ini menambahkan *generalization gap* DKT serta rangkuman perubahan utility PPO dari update awal hingga akhir. Nilai hilang tetap ditandai sebagai hilang; tidak dilakukan interpolasi atau perhitungan ulang checkpoint.

In [ ]:
if MODE == 'report':
    dkt_file=WORKSPACE/'dkt'/'full'/'dkt_history.csv'
    if dkt_file.is_file():
        dh=pd.read_csv(dkt_file)
        if {'epoch','train_batch_bce','val_bce'}.issubset(dh.columns):
            fig,ax=plt.subplots(figsize=(9.5,4.4))
            gap=dh['val_bce']-dh['train_batch_bce']
            ax.plot(dh['epoch'],gap,marker='o',label='Validation BCE − Training BCE')
            ax.axhline(0,color='black',ls='--',lw=1)
            best=dh.loc[dh['val_bce'].idxmin()]
            ax.axvline(best['epoch'],color='#E15759',ls=':',label=f'Epoch terpilih {int(best.epoch)}')
            ax.set(title='Generalization gap DKT per epoch',xlabel='Epoch',ylabel='Selisih BCE')
            ax.legend();ax.grid(alpha=.2)
            show_save(fig,'16_generalization_gap_dkt')
            pd.DataFrame({'epoch':dh['epoch'],'train_batch_bce':dh['train_batch_bce'],
                          'val_bce':dh['val_bce'],'gap_bce':gap}).to_csv(
                              LT/'dkt_generalization_gap.csv',index=False)
        else:
            print('Kolom DKT history tidak sesuai untuk gap:',list(dh.columns))
    fig,axs=plt.subplots(1,3,figsize=(15.5,4.2))
    rows=[]
    for model,h in histories.items():
        v=h.dropna(subset=['utility_val'])
        if v.empty:continue
        v=v.sort_values('update')
        axs[0].plot(v['update'],v['utility_val'],marker='o',markersize=3,
                    label=model_labels[model],color=figure_colors[model])
        axs[1].plot(v['update'],v['kc_available'],label=model_labels[model],color=figure_colors[model])
        axs[2].plot(v['update'],v['auc'],label=model_labels[model],color=figure_colors[model])
        rowbest=v.loc[v['utility_val'].idxmax()]
        rows.append({'model':model,'update_awal':int(v.iloc[0]['update']),
                     'update_terakhir':int(v.iloc[-1]['update']),
                     'update_terpilih':int(rowbest['update']),
                     'utility_awal':float(v.iloc[0]['utility_val']),
                     'utility_akhir':float(v.iloc[-1]['utility_val']),
                     'utility_terbaik':float(rowbest['utility_val'])})
        axs[0].scatter([rowbest['update']],[rowbest['utility_val']],s=48,color=figure_colors[model],edgecolors='black',zorder=4)
    for ax,(t,y) in zip(axs,[('Utility validasi','Utility'),('KC Coverage validasi','KC Available'),('AUC validasi','AUC')]):
        ax.set(title=t,xlabel='PPO update',ylabel=y);ax.grid(alpha=.2)
    axs[0].legend(fontsize=8)
    show_save(fig,'17_ppo_dinamika_checkpoint')
    trajectory=pd.DataFrame(rows)
    trajectory.to_csv(LT/'ringkasan_dinamika_ppo.csv',index=False)
    display(trajectory.round(5))
    display(Markdown('**Interpretasi.** Checkpoint terpilih menggunakan utility validasi terbesar. Jika pembelajaran setelah checkpoint terpilih menurunkan utility, hal itu menunjukkan bahwa PPO belum meningkatkan kebijakan secara monoton; tidak membuktikan kontribusi multi-agent secara kausal.'))

## 13.13 Stabilitas multi-seed dan margin terhadap baseline terbaik

Heatmap memperlihatkan **selisih MARL terhadap baseline terbaik pada seed yang sama**. Margin positif berarti MARL lebih baik; untuk BCE, tanda dibalik agar interpretasinya konsisten. Perbandingan tidak mencampur mahasiswa selection dan audit. Lima seed tidak dianggap sebagai lima sampel populasi yang sepenuhnya independen.

In [ ]:
if MODE == 'report':
    margin_rows=[]
    for (subset,seed),g in DATA['multiseed'].groupby(['subset','seed']):
        t=g.set_index('model')
        for metric in ['kc_available','auc','bce']:
            base=(t.loc[['rl_flat','rl_feat'],metric].min() if metric=='bce'
                  else t.loc[['rl_flat','rl_feat'],metric].max())
            raw_delta=float(t.loc['marl',metric]-base)
            margin_rows.append({'subset':subset,'seed':int(seed),'metric':metric,
                                'delta_asli':raw_delta,
                                'margin_keunggulan':-raw_delta if metric=='bce' else raw_delta})
    margins=pd.DataFrame(margin_rows)
    margins.to_csv(LT/'margin_marl_terhadap_baseline_per_seed.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(14,4.8))
    metrics=['kc_available','auc','bce']
    for ax,subset in zip(axes,['selection','audit']):
        t=margins[margins.subset==subset].pivot(index='seed',columns='metric',values='margin_keunggulan')
        t=t.reindex(columns=metrics)
        values=t.to_numpy(dtype=float)
        scale=np.maximum(np.max(np.abs(values),axis=0),1e-8)
        normalized=values / scale[None,:]
        im=ax.imshow(normalized,cmap='RdBu',vmin=-1,vmax=1,aspect='auto')
        ax.set_xticks(range(3),['KC Available','AUC','BCE (dibalik)'])
        ax.set_yticks(range(len(t.index)),[str(s) for s in t.index]);ax.set_ylabel('Seed')
        ax.set_title(f'Margin MARL vs baseline terbaik — {subset}')
        for ri in range(values.shape[0]):
            for ci in range(values.shape[1]):
                ax.text(ci,ri,f'{values[ri,ci]:+.3f}',ha='center',va='center',
                        color='white' if abs(normalized[ri,ci])>.60 else 'black',fontsize=9)
        fig.colorbar(im,ax=ax,shrink=.77,label='Margin per metrik (dinormalisasi)')
    show_save(fig,'18_heatmap_margin_multiseed')
    wins_long=margins.assign(unggul=lambda d:d.margin_keunggulan>0)
    won=wins_long.groupby(['subset','metric'])['unggul'].sum().unstack('metric').reindex(columns=metrics)
    won.to_csv(LT/'jumlah_kemenangan_per_metrik.csv')
    display(won)
    fig,axes=plt.subplots(1,2,figsize=(11.5,4.1),sharey=True)
    for ax,subset in zip(axes,['selection','audit']):
        bars=ax.bar(['KC Available','AUC','BCE'],won.loc[subset].values)
        ax.bar_label(bars,labels=[f'{int(x)}/5' for x in won.loc[subset].values],padding=2)
        ax.set_ylim(0,5.6);ax.set(title=f'Seed yang dimenangkan MARL — {subset}',ylabel='Jumlah seed')
        ax.grid(axis='y',alpha=.16)
    show_save(fig,'19_kemenangan_per_seed')
    display(Markdown('**Interpretasi.** Coverage MARL konsisten lebih tinggi pada seed yang diuji, tetapi margin AUC dan BCE dapat berubah tanda. Grafik ini membantu mengevaluasi kestabilan, bukan pengganti pengujian statistik pada mahasiswa independen.'))

## 13.14 Perubahan metrik pada studi ablasi

Selisih dihitung antara varian ablasi dan Full MARL pada seed yang sama. Untuk BCE, nilai asli dipertahankan dan diberi tanda arah penilaian: nilai negatif berarti perbaikan. Dua seed belum cukup untuk menyatakan efek komponen signifikan.

In [ ]:
if MODE == 'report':
    t=DATA['ablation']
    data=[]
    for (subset,seed),g in t.groupby(['subset','seed']):
        ref=g[g.model=='marl']
        if len(ref)!=1: raise RuntimeError('Full MARL harus ada tepat sekali per seed/subset ablasi.')
        ref=ref.iloc[0]
        for _,r in g.iterrows():
            if r.model=='marl':continue
            for metric in ['kc_available','auc','bce']:
                data.append({'subset':subset,'seed':int(seed),'variant':r.model,
                             'metric':metric,'delta_varian_minus_full':float(r[metric]-ref[metric])})
    delta_full=pd.DataFrame(data)
    delta_full.to_csv(LT/'ablation_delta_seluruh_seed.csv',index=False)
    labels_variant={
        'marl_no_planning':'No Planning',
        'marl_no_rank_reward':'No Rank Reward',
        'marl_no_coverage_reward':'No Coverage Reward',
        'marl_no_semantic_prior':'No Semantic Prior'}
    part=delta_full[delta_full.subset=='audit']
    means=part.groupby(['variant','metric'])['delta_varian_minus_full'].mean().unstack('metric')
    means=means.reindex(labels_variant).reindex(columns=['kc_available','auc','bce'])
    fig,ax=plt.subplots(figsize=(10.4,4.5))
    v=means.to_numpy(dtype=float)
    direction=v.copy();direction[:,2]=-direction[:,2]  # BCE lebih rendah lebih baik
    scale=np.maximum(np.max(np.abs(direction),axis=0),1e-8)
    normalized=direction / scale[None,:]
    im=ax.imshow(normalized,cmap='RdBu',vmin=-1,vmax=1,aspect='auto')
    ax.set_xticks(range(3),['Δ KC Available ↑','Δ AUC ↑','Δ BCE ↓'])
    ax.set_yticks(range(len(means)),[labels_variant[k] for k in means.index])
    ax.set_title('Perubahan rata-rata ablasi terhadap Full MARL — audit')
    for ri in range(v.shape[0]):
        for ci in range(v.shape[1]):
            ax.text(ci,ri,f'{v[ri,ci]:+.4f}',ha='center',va='center',
                    color='white' if abs(normalized[ri,ci])>.65 else 'black',fontsize=10)
    fig.colorbar(im,ax=ax,shrink=.85,label='Keunggulan per metrik (BCE dibalik)')
    show_save(fig,'20_heatmap_delta_ablasi')
    display(means.round(5))
    display(Markdown('**Interpretasi.** Penghilangan suatu komponen kadang memperbaiki satu metrik sambil menurunkan metrik lain. Pada kolom Δ BCE, nilai negatif justru lebih baik; perhatikan arah sumbu sebelum menafsirkan warna. Tidak ada klaim sebab-akibat yang terpisah sempurna dari perubahan fitur atau struktur model.'))

## 13.15 Ringkasan penelitian dan daftar hasil

Laporan otomatis menggabungkan temuan yang telah dihitung dari artefak eksperimen. Semua tabel/grafik dibuat di bawah `outputs/laporan/`, tanpa menimpa hasil training, checkpoint, manifest, atau prediksi final. Hasil pada test historis tidak digunakan untuk mengubah reward atau parameter kebijakan.

In [ ]:
if MODE == 'report':
    results=fin[['kc_available','kc_global','auc','bce']]
    bests={m: (results[m].idxmin() if m=='bce' else results[m].idxmax()) for m in results.columns}
    n_joint=int(win_agg.loc['audit','win_kc_auc_bce'])
    note=f"""# Ringkasan Hasil Penelitian Riset_MARL

## Dataset dan protokol
DBE-KT22; {IDENTITIES['full']['questions']} soal dan {IDENTITIES['full']['knowledge_components']} KC. Sebanyak {df_counts['train']} mahasiswa menjadi data training, {df_counts['val']} mahasiswa untuk validation (terbagi menjadi kelompok DKT, pemilihan kebijakan, dan audit), dan {df_counts['test']} mahasiswa berada dalam test historis. Panjang tes adaptif adalah {MANIFEST['config']['test_length']} soal. Evaluasi menggunakan replay respons yang telah tercatat.

## Hasil final test (seed 42)
- KC Available tertinggi: {model_labels[bests['kc_available']]} ({results.loc[bests['kc_available'],'kc_available']:.4f}).
- KC Global tertinggi: {model_labels[bests['kc_global']]} ({results.loc[bests['kc_global'],'kc_global']:.4f}).
- AUC tertinggi: {model_labels[bests['auc']]} ({results.loc[bests['auc'],'auc']:.4f}).
- BCE terendah: {model_labels[bests['bce']]} ({results.loc[bests['bce'],'bce']:.4f}).

## Multi-seed
Pada policy audit, MARL unggul atas kedua baseline pada ketiga metrik secara bersamaan dalam {n_joint}/5 seed. Hasil ini menggambarkan variabilitas pembelajaran, bukan signifikansi populasi.

## Uji statistik
Paired bootstrap CI 95% dan Wilcoxon-H﻿olm dibaca dari berkas final test, bukan dihitung ulang untuk melakukan pemilihan model. Interval KC Available terhadap kedua baseline tidak mencakup nol, sedangkan interval AUC dan BCE masih mencakup nol.

## Batasan
Pengambilan soal menggunakan replay offline; respons mahasiswa untuk soal yang tidak pernah dijawab tidak tersedia. Kelompok final test telah digunakan dalam penelitian sebelumnya dan oleh karena itu tidak diklaim sebagai holdout yang sepenuhnya baru. Seluruh analisis merupakan interpretasi atas hasil yang telah dibekukan, bukan eksperimen training tambahan.
"""
    (LR/'ringkasan_penelitian.md').write_text(note,encoding='utf8')
    inventory=pd.DataFrame([{'file':str(x.relative_to(LAPORAN)),'ukuran_kb':round(x.stat().st_size/1024,1)}
                            for x in sorted(LAPORAN.rglob('*')) if x.is_file()])
    inventory.to_csv(LT/'daftar_artefak_laporan.csv',index=False)
    display(Markdown(note))
    display(inventory)
    print(f'LAPORAN SELESAI — {len(inventory)} file tersimpan di {LAPORAN}')
    print('Tidak ada DKT/PPO training atau final test ulang pada mode report.')

# 14. Pelaporan publik dari data agregat

Bagian ini dijalankan dengan `MODE='public'` dan hanya membaca berkas CSV agregat yang dibagikan di folder `results/public/`. Tidak diperlukan dataset DBE-KT22 asli, checkpoint, prediksi per mahasiswa, atau akses Google Drive. Seluruh visualisasi ditampilkan di notebook dan disimpan pada `outputs/laporan_public/`.

**Batas reproduksi:** mode ini mereproduksi penyajian hasil dari tabel agregat yang telah dipublikasikan, **bukan** pelatihan atau penilaian ulang model. Untuk mereproduksi eksperimen dari awal, diperlukan izin akses dataset asli dan mode `smoke`/`full`/`multiseed` sesuai protokol yang didokumentasikan.


In [ ]:
if MODE == 'public':
    from IPython.display import display, Markdown, Image
    PUB = ROOT / 'results' / 'public'
    PT = PUB / 'tables'
    PF = PUB / 'figures'
    OUT_PUB = WORKSPACE / 'laporan_public'
    (OUT_PUB/'figures').mkdir(parents=True,exist_ok=True)
    (OUT_PUB/'tables').mkdir(parents=True,exist_ok=True)
    needed = ['final_test_comparison.csv','paired_statistics.csv',
              'full_model_comparison.csv','multiseed_model_comparison.csv','ablation_model_comparison.csv']
    missing = [str(PT/x) for x in needed if not (PT/x).is_file()]
    if missing:
        raise FileNotFoundError('Berkas agregat publik belum tersedia: '+str(missing)+
           '. Clone repository GitHub ke /content/Riset_MARL atau tetapkan RISET_MARL_REPO_DIR pada sel awal.')
    PUB_DATA = {name:pd.read_csv(PT/name) for name in needed}
    final_public=PUB_DATA['final_test_comparison.csv'].set_index('model').reindex(['rl_flat','rl_feat','marl'])
    display(Markdown('### 14.1 Hasil final test yang dilaporkan'))
    display(final_public.round(5))
    display(Markdown('### 14.2 Statistik pasangan dari eksperimen yang dibekukan'))
    display(PUB_DATA['paired_statistics.csv'].round(5))
    display(Markdown('Data ini adalah hasil agregat penelitian. Kelompok test DBE-KT22 telah terpapar pada studi sebelumnya; bukan holdout baru.'))


## 14.3 Penyajian ulang kurva dan perbandingan

Grafik dibuat langsung dari tabel yang dirilis, tanpa membuka prediksi mahasiswa. Posisi dan interval dihitung dari ringkasan yang tersedia. Karena hanya terdapat lima seed validasi dan dua seed ablasi, grafik deviasi standar tidak boleh dianggap sebagai pembuktian signifikansi populasi.


In [ ]:
if MODE == 'public':
    labels = {'rl_flat':'RL-Flat','rl_feat':'RL-Feat','marl':'MARL'}
    columns = [('kc_available','KC Coverage (↑)'),('kc_global','KC Global (↑)'),('auc','AUC (↑)'),('bce','BCE (↓)')]
    fig,axes=plt.subplots(2,2,figsize=(11,7),constrained_layout=True)
    for (metric,heading),ax in zip(columns,axes.flat):
        vals=final_public[metric]
        ax.bar([labels[x] for x in vals.index],vals.values)
        ax.set_title(heading)
        ax.grid(axis='y',alpha=.2)
        for i,v in enumerate(vals.values):ax.text(i,v,f'{v:.4f}',ha='center',va='bottom',fontsize=9)
    fig.suptitle('Final test (157 mahasiswa, seed 42) — hasil agregat')
    fig.savefig(OUT_PUB/'figures'/'final_test_public.png',dpi=170,bbox_inches='tight')
    fig.savefig(OUT_PUB/'figures'/'final_test_public.svg',bbox_inches='tight')
    plt.show();plt.close(fig)
    ms=PUB_DATA['multiseed_model_comparison.csv']
    audit=ms[ms.subset=='audit']
    agg=audit.groupby('model')[['kc_available','auc','bce']].agg(['mean','std'])
    agg.to_csv(OUT_PUB/'tables'/'multiseed_mean_std_public.csv')
    display(Markdown('### Rata-rata ± SD pada policy audit (lima seed)'))
    display(agg.round(5))
    fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
    for metric,ax in zip(['kc_available','auc','bce'],axes):
        means=agg[(metric,'mean')].reindex(['rl_flat','rl_feat','marl'])
        sd=agg[(metric,'std')].reindex(['rl_flat','rl_feat','marl'])
        ax.bar([labels[x] for x in means.index],means.values,yerr=sd.values,capsize=4)
        ax.set_title(metric);ax.grid(axis='y',alpha=.2)
    fig.suptitle('Policy audit — rata-rata dan variasi lima seed')
    fig.savefig(OUT_PUB/'figures'/'multiseed_public.png',dpi=170,bbox_inches='tight')
    fig.savefig(OUT_PUB/'figures'/'multiseed_public.svg',bbox_inches='tight')
    plt.show();plt.close(fig)
    ab=PUB_DATA['ablation_model_comparison.csv']
    ab_audit=ab[ab.subset=='audit'].groupby('model')[['kc_available','auc','bce']].mean()
    ab_audit.to_csv(OUT_PUB/'tables'/'ablation_audit_mean_public.csv')
    display(Markdown('### Studi ablasi — rata-rata dua seed pada policy audit'))
    display(ab_audit.round(5))


## 14.4 Grafik pendukung dan interpretasi

Gambar arsitektur, kurva DKT/PPO, visualisasi ROC/kalibrasi, dan interval kepercayaan merupakan artefak agregat dari eksperimen terdahulu. Pada mode publik, gambar tersebut ditampilkan tanpa membaca data individu. AUC dan BCE final tidak menunjukkan perbedaan yang didukung confidence interval 95%; bukti peningkatan paling kuat terdapat pada KC Coverage.


In [ ]:
if MODE == 'public':
    for title,filename in [
        ('Arsitektur DKT–MARL','02_arsitektur_dkt_marl.png'),
        ('Kurva training DKT','03_kurva_dkt.png'),
        ('Kurva PPO seed 42','04_kurva_ppo_full_seed42.png'),
        ('Studi ablasi','07_ablation_audit.png'),
        ('Bootstrap CI 95%','09_bootstrap_ci_final.png'),
        ('ROC dan kalibrasi (agregat)','10_roc_dan_kalibrasi.png')]:
        p=PF/filename
        if p.is_file():
            display(Markdown('### '+title))
            display(Image(filename=str(p),width=780))
    best_kc=final_public.kc_available.idxmax()
    best_auc=final_public.auc.idxmax()
    best_bce=final_public.bce.idxmin()
    note=(f'Final test: KC Available tertinggi oleh {labels[best_kc]} '
          f'({final_public.loc[best_kc,"kc_available"]:.4f}); '
          f'AUC tertinggi oleh {labels[best_auc]} '
          f'({final_public.loc[best_auc,"auc"]:.4f}); '
          f'BCE terendah oleh {labels[best_bce]} '
          f'({final_public.loc[best_bce,"bce"]:.4f}). '
          'Klaim generalisasi dibatasi pada replay offline DBE-KT22 dan kelompok test historis.')
    (OUT_PUB/'ringkasan_public.md').write_text('# Ringkasan Pelaporan Publik\n\n'+note+'\n',encoding='utf-8')
    display(Markdown('**Interpretasi:** '+note))
    print('Laporan agregat publik selesai. File baru:',OUT_PUB)


## 14.5 Heatmap evaluasi berbasis tabel publik

Bagian ini memvisualisasikan margin per seed dan selisih ablasi langsung dari CSV agregat yang dirilis. Data yang digunakan adalah **hasil lama yang sudah dibekukan**, bukan training atau pengujian ulang. Grafik distribusi interaksi memerlukan ZIP asli DBE-KT22 dan hanya dibuat dalam mode `report` di Drive pribadi; tidak ditampilkan dalam mode publik jika agregat dataset belum disetujui untuk disebarkan.

In [ ]:
if MODE == 'public':
    ms=PUB_DATA['multiseed_model_comparison.csv']
    ab=PUB_DATA['ablation_model_comparison.csv']
    import matplotlib.pyplot as plt
    table_rows=[]
    for (subset,seed),g in ms.groupby(['subset','seed']):
        t=g.set_index('model')
        for metric in ['kc_available','auc','bce']:
            baseline = (t.loc[['rl_flat','rl_feat'],metric].min() if metric=='bce'
                        else t.loc[['rl_flat','rl_feat'],metric].max())
            raw=float(t.loc['marl',metric]-baseline)
            table_rows.append({'subset':subset,'seed':int(seed),'metric':metric,
                               'margin_positif_baik':-raw if metric=='bce' else raw})
    pub_margin=pd.DataFrame(table_rows)
    pub_margin.to_csv(OUT_PUB/'tables'/'margin_marl_per_seed.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(12.5,4.5),constrained_layout=True)
    for ax,subset in zip(axs,['selection','audit']):
        t=pub_margin[pub_margin.subset==subset].pivot(index='seed',columns='metric',values='margin_positif_baik')
        t=t.reindex(columns=['kc_available','auc','bce'])
        v=t.to_numpy(dtype=float)
        scale=np.maximum(np.max(np.abs(v),axis=0),1e-8)
        normalized=v/scale[None,:]
        im=ax.imshow(normalized,cmap='RdBu',vmin=-1,vmax=1,aspect='auto')
        ax.set_xticks(range(3),['KC Available','AUC','BCE (dibalik)'])
        ax.set_yticks(range(len(t)),[str(x) for x in t.index]);ax.set(title=f'Margin MARL — {subset}',ylabel='Seed')
        for i in range(v.shape[0]):
            for j in range(v.shape[1]):
                ax.text(j,i,f'{v[i,j]:+.3f}',ha='center',va='center',fontsize=9,
                        color='white' if abs(normalized[i,j])>.60 else 'black')
        fig.colorbar(im,ax=ax,shrink=.75)
    fig.savefig(OUT_PUB/'figures'/'margin_multiseed_public.png',dpi=180,bbox_inches='tight')
    fig.savefig(OUT_PUB/'figures'/'margin_multiseed_public.svg',bbox_inches='tight')
    display(fig);plt.close(fig)

    abrows=[]
    for (subset,seed),g in ab.groupby(['subset','seed']):
        ref=g[g.model=='marl'].iloc[0]
        for _,r in g.iterrows():
            if r.model=='marl':continue
            for metric in ['kc_available','auc','bce']:
                abrows.append({'subset':subset,'seed':int(seed),'variant':r.model,
                               'metric':metric,'delta':float(r[metric]-ref[metric])})
    pub_deltas=pd.DataFrame(abrows)
    pub_deltas.to_csv(OUT_PUB/'tables'/'ablation_delta_per_seed.csv',index=False)
    d=pub_deltas[pub_deltas.subset=='audit'].groupby(['variant','metric'])['delta'].mean().unstack()
    d=d.reindex(columns=['kc_available','auc','bce'])
    fig,ax=plt.subplots(figsize=(9.5,4.2))
    v=d.to_numpy(dtype=float)
    favorable=v.copy();favorable[:,2]=-favorable[:,2]
    scale=np.maximum(np.max(np.abs(favorable),axis=0),1e-8)
    normalized=favorable/scale[None,:]
    im=ax.imshow(normalized,cmap='RdBu',vmin=-1,vmax=1,aspect='auto')
    ax.set_xticks(range(3),['Δ KC Available ↑','Δ AUC ↑','Δ BCE ↓'])
    ax.set_yticks(range(len(d)),[str(x).removeprefix('marl_').replace('_',' ') for x in d.index])
    ax.set_title('Ablasi — rata-rata perubahan terhadap Full MARL (audit)')
    for i in range(v.shape[0]):
        for j in range(v.shape[1]):
            ax.text(j,i,f'{v[i,j]:+.4f}',ha='center',va='center',fontsize=9,
                    color='white' if abs(normalized[i,j])>.65 else 'black')
    fig.colorbar(im,ax=ax,shrink=.8)
    fig.savefig(OUT_PUB/'figures'/'ablation_delta_public.png',dpi=180,bbox_inches='tight')
    fig.savefig(OUT_PUB/'figures'/'ablation_delta_public.svg',bbox_inches='tight')
    display(fig);plt.close(fig)
    display(Markdown('**Catatan:** warna distandarkan di masing-masing kolom metrik dan angka tetap selisih asli. Margin positif pada heatmap seed menandakan MARL unggul terhadap baseline terbaik. Heatmap ablasi menampilkan selisih metrik asli: Δ BCE negatif lebih baik. Diagram tidak membuktikan signifikansi maupun kausalitas.'))

# 15. Audit kualitas laporan penelitian

## 15.1 Pemeriksaan integritas dan reproduksibilitas publik

Pemeriksaan berikut memverifikasi checksum berkas publik, kesesuaian hasil seed yang seharusnya identik, kelengkapan metrik, serta perbedaan final MARL terhadap kedua baseline. Prosedur ini dijalankan **tanpa dataset mentah, checkpoint, training ulang, atau pengujian ulang pada mahasiswa**. Pemeriksaan ini tidak membuktikan kebaruan metode ataupun keunggulan statistik pada ketiga metrik.

In [ ]:
if MODE == 'public':
    import hashlib, json
    from IPython.display import display, Markdown
    checks=[]
    manifest=json.loads((PUB/'PROVENANCE.json').read_text(encoding='utf8'))
    for rel,expected in manifest['sha256'].items():
        path=PUB/rel
        assert path.is_file(), f'Berkas agregat hilang: {rel}'
        assert hashlib.sha256(path.read_bytes()).hexdigest()==expected, f'Checksum berbeda: {rel}'
    checks.append(('Integritas SHA-256',f"{len(manifest['sha256'])} berkas",'LULUS'))
    fin=PUB_DATA['final_test_comparison.csv'].set_index('model')
    assert set(fin.index)=={'rl_flat','rl_feat','marl'}
    for metric in ('kc_available','kc_global','auc'):
        assert fin[metric].between(0,1).all(),metric
    assert (fin.bce>=0).all() and (fin.length==20).all()
    checks.append(('Validitas skema final','3 model, 20 soal','LULUS'))
    full=pd.read_csv(PT/'full_model_comparison.csv')
    multi=PUB_DATA['multiseed_model_comparison.csv']
    abl=PUB_DATA['ablation_model_comparison.csv']
    assert len(full)==6 and len(multi)==30 and len(abl)==20
    assert set(multi.seed)=={42,123,321,2026,2027}
    assert set(abl.seed)=={42,123}
    assert not multi.duplicated(['seed','model','subset']).any()
    assert not abl.duplicated(['seed','model','subset']).any()
    cols=['kc_available','kc_global','auc','bce','length']
    keys=['seed','model','subset']
    def matching(a,b):
        a=a[keys+cols].sort_values(keys).reset_index(drop=True)
        b=b[keys+cols].sort_values(keys).reset_index(drop=True)
        assert a[keys].equals(b[keys]),'Kunci hasil berbeda'
        assert np.allclose(a[cols],b[cols],atol=1e-10,rtol=0),'Metrik hasil berbeda'
    matching(full,multi[multi.seed==42])
    matching(abl[abl.model=='marl'],multi[(multi.model=='marl')&multi.seed.isin([42,123])])
    checks.append(('Konsistensi full/multi-seed/ablasi','seed 42 dan 123','LULUS'))
    stats=PUB_DATA['paired_statistics.csv']
    assert len(stats)==6 and not stats.duplicated(['comparison','metric']).any()
    for row in stats.itertuples(index=False):
        baseline=row.comparison.removeprefix('marl vs ')
        delta=fin.loc['marl',row.metric]-fin.loc[baseline,row.metric]
        assert abs(delta-row.difference)<1e-9
        assert row.bootstrap_ci_low<=row.bootstrap_ci_high
        assert 0<=row.wilcoxon_p<=1 and 0<=row.wilcoxon_p_holm<=1
    checks.append(('Kesesuaian selisih final dan statistik','6 perbandingan','LULUS'))
    results=pd.DataFrame(checks,columns=['pemeriksaan','cakupan','status'])
    loc=WORKSPACE/'laporan_public'
    (loc/'tables').mkdir(parents=True,exist_ok=True)
    (loc/'reports').mkdir(parents=True,exist_ok=True)
    results.to_csv(loc/'tables'/'audit_publik.csv',index=False)
    (loc/'reports'/'audit_publik.json').write_text(json.dumps({'status':'LULUS','batas':'Hanya integritas agregat publik; bukan validasi kausal atau test baru.'},ensure_ascii=False,indent=2),encoding='utf8')
    display(Markdown('**Audit konsistensi agregat publik berhasil.**'))
    display(results)
elif MODE == 'report':
    print('Mode report: integritas dataset/split dan checkpoint diperiksa di Bagian 13.1.')
else:
    print('Audit publik dilewati: MODE=',MODE)


## 15.2 Catatan penggunaan dalam tesis

- **KC Coverage:** keunggulan final MARL didukung perbedaan yang konsisten dalam evaluasi pasangan. KC Available dan KC Global mempunyai penyebut berbeda.
- **AUC pooled:** RL-Flat sedikit lebih tinggi dari MARL pada final test. Wilcoxon AUC per mahasiswa tidak mempunyai estimand identik dengan AUC pooled.
- **BCE pooled:** MARL memiliki nilai numerik terendah; confidence interval selisih dengan baseline masih mencakup nol.
- **Multi-seed dan ablasi:** hasil pada kelompok mahasiswa yang sama bukan replikasi data independen; lima seed multi-seed dan dua seed ablasi membatasi klaim kestabilan.
- **Kebaruan dan fairness:** arsitektur tiga aktor kooperatif harus dibahas dengan membedakan efek multi-agent dari fitur, prior, dan kapasitas model.

Baca `docs/PENILAIAN_TESIS.md` dan `docs/STATISTIK_DAN_KLAIM.md` sebelum menggunakan grafik dalam Bab IV dan Bab V.